In [5]:
import sys
print(sys.executable)

C:\Users\manas\OneDrive\Desktop\KJSCE\amazon_ml_26\amazon-ml-challenge-2026\.venv\Scripts\python.exe


In [6]:
import pandas
import numpy
import rapidfuzz

print("Environment OK")

Environment OK


In [7]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import normalize_name, normalize_address
from src.features import feature_pair

DATA_DIR = Path(r"C:\Users\manas\Downloads\train")

print("Environment ready")
print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("Pandas:", pd.__version__)

Environment ready
ROOT: C:\Users\manas\OneDrive\Desktop\KJSCE\amazon_ml_26\amazon-ml-challenge-2026
DATA_DIR: C:\Users\manas\Downloads\train
Pandas: 2.3.3


In [8]:
DATA_DIR = Path(r"C:\Users\manas\Downloads\train")

print("Data directory:", DATA_DIR)
print("Exists:", DATA_DIR.exists())

for filename in [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
]:
    path = DATA_DIR / filename
    print(filename, "->", path.exists())

Data directory: C:\Users\manas\Downloads\train
Exists: True
train_source1.tsv -> True
train_source2.tsv -> True
train_source3.tsv -> True
train_ground_truth.tsv -> True


In [9]:
s1 = pd.read_csv(
    DATA_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str,
    nrows=10_000,
    keep_default_na=False,
)

s2 = pd.read_csv(
    DATA_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str,
    nrows=10_000,
    keep_default_na=False,
)

s3 = pd.read_csv(
    DATA_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str,
    nrows=10_000,
    keep_default_na=False,
)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

S1: (10000, 4)
S2: (10000, 4)
S3: (10000, 4)


In [10]:
print("S1 columns:")
print(s1.columns.tolist())

print("\nS2 columns:")
print(s2.columns.tolist())

print("\nS3 columns:")
print(s3.columns.tolist())

S1 columns:
['entity_id', 'business_name', 'business_address', 'country']

S2 columns:
['entity_id', 'business_name', 'business_address', 'country']

S3 columns:
['entity_id', 'business_name', 'business_address', 'country']


In [11]:
display(s1.head(3))
display(s2.head(3))
display(s3.head(3))

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US


In [12]:
gt = pd.read_csv(
    DATA_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str,
    nrows=10_000,
    keep_default_na=False,
)

print("GT shape:", gt.shape)
print("GT columns:", gt.columns.tolist())

display(gt.head(10))

GT shape: (10000, 2)
GT columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [13]:
def prepare_source(df):
    df = df.copy()

    df["name_norm"] = df["business_name"].map(normalize_name)
    df["address_norm"] = df["business_address"].map(normalize_address)

    return df


s1_p = prepare_source(s1)
s2_p = prepare_source(s2)
s3_p = prepare_source(s3)

In [14]:
display(
    s1_p[
        [
            "business_name",
            "name_norm",
            "business_address",
            "address_norm",
            "country",
        ]
    ].head(10)
)

,business_name,name_norm,business_address,address_norm,country
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US
2,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US
8,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US


In [15]:
s2_lookup = s2_p.set_index("entity_id").to_dict("index")
s3_lookup = s3_p.set_index("entity_id").to_dict("index")

In [16]:
s1_lookup = s1_p.set_index("entity_id").to_dict("index")

In [17]:
display(
    gt[
        ["source1_entity_id", "matched_entity_ids"]
    ].head(20)
)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [18]:
for _, row in gt.head(10).iterrows():
    print("S1:", row["source1_entity_id"])
    print("Matches:", row["matched_entity_ids"])
    print()

S1: S1-965667
Matches: S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364

S1: S1-55344266
Matches: S2-249013014,S2-197070651,S3-478195123,S3-384364074

S1: S1-343815751
Matches: S2-790675320,S2-479876582,S3-878454467

S1: S1-656753428
Matches: S2-153058913,S2-24659151,S3-679606215

S1: S1-102811957
Matches: S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785

S1: S1-18727616
Matches: S2-755677256,S3-187831601,S3-641489370,S3-476250621

S1: S1-318373630
Matches: S2-660036492,S3-804600254

S1: S1-86989137
Matches: S3-274817120,S3-312496301

S1: S1-29845983
Matches: S2-648035184,S3-588502663

S1: S1-789009573
Matches: S2-383871912,S3-74481402,S3-576451439



In [19]:
print("S1:", s1.columns.tolist())
print("S2:", s2.columns.tolist())
print("S3:", s3.columns.tolist())
print("GT:", gt.columns.tolist())

S1: ['entity_id', 'business_name', 'business_address', 'country']
S2: ['entity_id', 'business_name', 'business_address', 'country']
S3: ['entity_id', 'business_name', 'business_address', 'country']
GT: ['source1_entity_id', 'matched_entity_ids']


In [20]:
# Load ONLY entity IDs from the full training sources.
# This is much smaller than loading all business fields.

s1_ids = pd.read_csv(
    DATA_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str,
    usecols=["entity_id"],
    keep_default_na=False,
)

s2_ids = pd.read_csv(
    DATA_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str,
    usecols=["entity_id"],
    keep_default_na=False,
)

s3_ids = pd.read_csv(
    DATA_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str,
    usecols=["entity_id"],
    keep_default_na=False,
)

print("S1 IDs:", len(s1_ids))
print("S2 IDs:", len(s2_ids))
print("S3 IDs:", len(s3_ids))

S1 IDs: 2206821
S2 IDs: 5034616
S3 IDs: 5285603


In [22]:
s1_id_set = set(s1_ids["entity_id"])
s2_id_set = set(s2_ids["entity_id"])
s3_id_set = set(s3_ids["entity_id"])

print("Unique S1 IDs:", len(s1_id_set))
print("Unique S2 IDs:", len(s2_id_set))
print("Unique S3 IDs:", len(s3_id_set))

Unique S1 IDs: 2206821
Unique S2 IDs: 5034616
Unique S3 IDs: 5285603


In [23]:
gt_full = pd.read_csv(
    DATA_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

print("GT rows:", len(gt_full))
display(gt_full.head())

GT rows: 2206821


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [24]:
gt_work = gt_full.copy()

gt_work = gt_work[
    gt_work["source1_entity_id"].isin(s1_id_set)
].copy()

gt_work = gt_work[
    gt_work["matched_entity_ids"].ne("")
].copy()

print("GT rows with S1 present and non-empty matches:", len(gt_work))

GT rows with S1 present and non-empty matches: 2083574


In [25]:
links = (
    gt_work[
        ["source1_entity_id", "matched_entity_ids"]
    ]
    .assign(
        matched_entity_id=lambda x:
            x["matched_entity_ids"].str.split(",")
    )
    .explode("matched_entity_id")
)

links["matched_entity_id"] = (
    links["matched_entity_id"]
    .str.strip()
)

print("Expanded GT links:", len(links))

Expanded GT links: 7638365


In [26]:
links["source"] = np.where(
    links["matched_entity_id"].isin(s2_id_set),
    "S2",
    np.where(
        links["matched_entity_id"].isin(s3_id_set),
        "S3",
        "UNAVAILABLE"
    )
)

In [27]:
positive_links = links[
    links["source"].isin(["S2", "S3"])
].copy()

positive_links = positive_links.rename(
    columns={
        "source1_entity_id": "s1_id",
        "matched_entity_id": "matched_id",
    }
)[
    ["s1_id", "matched_id", "source"]
]

print("Valid positive links:", len(positive_links))
display(positive_links.head(20))

Valid positive links: 7638365


,s1_id,matched_id,source
0,S1-965667,S2-681193310,S2
0,S1-965667,S2-743505751,S2
0,S1-965667,S3-775321672,S3
0,S1-965667,S3-11291185,S3
0,S1-965667,S3-860443364,S3
1,S1-55344266,S2-249013014,S2
1,S1-55344266,S2-197070651,S2
1,S1-55344266,S3-478195123,S3
1,S1-55344266,S3-384364074,S3
2,S1-343815751,S2-790675320,S2


In [28]:
print(
    positive_links["source"].value_counts()
)

source
S3    3944746
S2    3693619
Name: count, dtype: int64


In [29]:
print("S1 IDs:", len(s1_id_set))
print("S2 IDs:", len(s2_id_set))
print("S3 IDs:", len(s3_id_set))

print("\nExample S2 IDs:")
print(list(s2_id_set)[:10])

print("\nExample S3 IDs:")
print(list(s3_id_set)[:10])

print("\nExample GT matched IDs:")
print(
    links["matched_entity_id"].head(20).tolist()
)

S1 IDs: 2206821
S2 IDs: 5034616
S3 IDs: 5285603

Example S2 IDs:
['S2-267735162', 'S2-137252383', 'S2-384672177', 'S2-104382426', 'S2-969413427', 'S2-436260564', 'S2-173046117', 'S2-713624589', 'S2-178099205', 'S2-999197530']

Example S3 IDs:
['S3-552895673', 'S3-11368916', 'S3-312869351', 'S3-203288673', 'S3-790893844', 'S3-184220961', 'S3-220304123', 'S3-334389126', 'S3-402298691', 'S3-476706302']

Example GT matched IDs:
['S2-681193310', 'S2-743505751', 'S3-775321672', 'S3-11291185', 'S3-860443364', 'S2-249013014', 'S2-197070651', 'S3-478195123', 'S3-384364074', 'S2-790675320', 'S2-479876582', 'S3-878454467', 'S2-153058913', 'S2-24659151', 'S3-679606215', 'S2-478959098', 'S2-553508714', 'S2-625774905', 'S3-728090388', 'S3-928796641']


In [30]:
s2_mask = links["matched_entity_id"].isin(s2_id_set)
s3_mask = links["matched_entity_id"].isin(s3_id_set)

print("In S2:", s2_mask.sum())
print("In S3:", s3_mask.sum())
print("In neither:", (~(s2_mask | s3_mask)).sum())

In S2: 3693619
In S3: 3944746
In neither: 0


In [31]:
overlap = s2_id_set & s3_id_set

print("S2 IDs:", len(s2_id_set))
print("S3 IDs:", len(s3_id_set))
print("S2/S3 overlapping IDs:", len(overlap))

S2 IDs: 5034616
S3 IDs: 5285603
S2/S3 overlapping IDs: 0


In [32]:
positive_counts = (
    positive_links
    .groupby(["s1_id", "source"])
    .size()
    .unstack(fill_value=0)
)

positive_counts["total"] = positive_counts.sum(axis=1)

print("S1 entities with positive links:", len(positive_counts))
print("\nPositive links by source:")
print(positive_links["source"].value_counts())

print("\nMatches per S1:")
print(positive_counts["total"].describe())

print("\nS1 entities by number of matches:")
print(positive_counts["total"].value_counts().sort_index())

S1 entities with positive links: 2083574

Positive links by source:
source
S3    3944746
S2    3693619
Name: count, dtype: int64

Matches per S1:
count    2.083574e+06
mean     3.665992e+00
std      1.526294e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: total, dtype: float64

S1 entities by number of matches:
total
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64


In [33]:
# Sample positive pairs for feature inspection
positive_sample = positive_links.sample(
    n=2000,
    random_state=42
).reset_index(drop=True)

positive_sample.head()

,s1_id,matched_id,source
0,S1-341462203,S2-219553948,S2
1,S1-783955316,S3-656241763,S3
2,S1-313221336,S3-872559960,S3
3,S1-516394603,S2-642842175,S2
4,S1-887045319,S2-612158384,S2


In [34]:
# Load only the rows needed for our sampled positive IDs

s1_needed = set(positive_sample["s1_id"])

s2_needed = set(
    positive_sample.loc[
        positive_sample["source"] == "S2",
        "matched_id"
    ]
)

s3_needed = set(
    positive_sample.loc[
        positive_sample["source"] == "S3",
        "matched_id"
    ]
)

print("S1 needed:", len(s1_needed))
print("S2 needed:", len(s2_needed))
print("S3 needed:", len(s3_needed))

S1 needed: 1998
S2 needed: 965
S3 needed: 1035


In [35]:
def load_rows_by_ids(path, needed_ids, chunksize=200_000):
    pieces = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize,
    ):
        matched = chunk[chunk["entity_id"].isin(needed_ids)]

        if not matched.empty:
            pieces.append(matched)

        if sum(len(x) for x in pieces) >= len(needed_ids):
            break

    if not pieces:
        return pd.DataFrame()

    return pd.concat(pieces, ignore_index=True)


s1_sample = load_rows_by_ids(
    DATA_DIR / "train_source1.tsv",
    s1_needed
)

s2_sample = load_rows_by_ids(
    DATA_DIR / "train_source2.tsv",
    s2_needed
)

s3_sample = load_rows_by_ids(
    DATA_DIR / "train_source3.tsv",
    s3_needed
)

print("S1 loaded:", len(s1_sample))
print("S2 loaded:", len(s2_sample))
print("S3 loaded:", len(s3_sample))

S1 loaded: 1998
S2 loaded: 965
S3 loaded: 1035


In [36]:
def prepare_source(df):
    df = df.copy()
    df["name_norm"] = df["business_name"].map(normalize_name)
    df["address_norm"] = df["business_address"].map(normalize_address)
    return df


s1_sample = prepare_source(s1_sample)
s2_sample = prepare_source(s2_sample)
s3_sample = prepare_source(s3_sample)

print(s1_sample[[
    "entity_id",
    "business_name",
    "business_address",
    "country",
    "name_norm",
    "address_norm"
]].head())

      entity_id                        business_name  \
0    S1-6232617                      Aspen Ridge LLC   
1  S1-277346734                  Ridge Detailing Inc   
2  S1-951451939      Maa Investments Private Limited   
3  S1-411744587  Vinita Herrell Sterling Managed LLC   
4  S1-804210432    Keystone Creative Associates PLLC   

                                    business_address country  \
0             TX, 834 Sunshine Medley Lane, Richmond      US   
1                  Rumford, 660 Penobscot Street, ME      US   
2  Villa No 68, Cyprus Palms, Serilingampally, K....   India   
3  2201 Chesapeake Harbour Drive, Fl 0, Annapolis...      US   
4                    211 Woodway Avenue, Spokane, WA      US   

                             name_norm  \
0                      aspen ridge llc   
1                  ridge detailing inc   
2      maa investments private limited   
3  vinita herrell sterling managed llc   
4    keystone creative associates pllc   

                         

In [37]:
s1_lookup = s1_sample.set_index("entity_id").to_dict("index")
s2_lookup = s2_sample.set_index("entity_id").to_dict("index")
s3_lookup = s3_sample.set_index("entity_id").to_dict("index")

print("Lookup sizes:")
print("S1:", len(s1_lookup))
print("S2:", len(s2_lookup))
print("S3:", len(s3_lookup))

Lookup sizes:
S1: 1998
S2: 965
S3: 1035


In [38]:
feature_rows = []

for _, pair in positive_sample.iterrows():
    s1_row = s1_lookup[pair["s1_id"]]

    if pair["source"] == "S2":
        match_row = s2_lookup[pair["matched_id"]]
    else:
        match_row = s3_lookup[pair["matched_id"]]

    features = feature_pair(s1_row, match_row)

    features["s1_id"] = pair["s1_id"]
    features["matched_id"] = pair["matched_id"]
    features["source"] = pair["source"]
    features["label"] = 1

    feature_rows.append(features)

positive_features = pd.DataFrame(feature_rows)

print("Feature rows:", len(positive_features))
print("Feature columns:")
print(positive_features.columns.tolist())

Feature rows: 2000
Feature columns:
['name_exact', 'name_ratio', 'name_token_set', 'name_token_sort', 'name_jaccard', 'name_char_jaccard', 'name_length_difference', 'name_length_ratio', 'address_exact', 'address_ratio', 'address_token_set', 'address_token_sort', 'address_jaccard', 'address_char_jaccard', 'address_length_difference', 'address_length_ratio', 'digit_overlap', 'country_same', 'both_exact', 's1_id', 'matched_id', 'source', 'label']


In [39]:
positive_features.head().T

,0,1,2,3,4
name_exact,0,1,1,0,0
name_ratio,0.21875,1.0,1.0,0.933333,0.948718
name_token_set,0.258065,1.0,1.0,0.933333,0.666667
name_token_sort,0.21875,1.0,1.0,0.933333,0.666667
name_jaccard,0.066667,1.0,1.0,0.6,0.428571
name_char_jaccard,0.054545,1.0,1.0,0.576923,0.731707
name_length_difference,2,0,0,1,2
name_length_ratio,0.939394,1.0,1.0,0.956522,0.95
address_exact,0,0,0,0,0
address_ratio,0.84,0.45,0.760563,0.765432,0.892308


In [40]:
positive_features.describe().T

,count,mean,std,min,25%,50%,75%,max
name_exact,2000.0,0.221000,0.415024,0.000000,0.000000,0.000000,0.000000,1.0
name_ratio,2000.0,0.804384,0.231938,0.064516,0.740487,0.875000,0.962963,1.0
name_token_set,2000.0,0.874260,0.241243,0.066667,0.887978,1.000000,1.000000,1.0
name_token_sort,2000.0,0.806790,0.241764,0.064516,0.738963,0.892857,1.000000,1.0
name_jaccard,2000.0,0.618253,0.329361,0.000000,0.500000,0.666667,1.000000,1.0
name_char_jaccard,2000.0,0.663982,0.285356,0.000000,0.500000,0.709005,0.878788,1.0
name_length_difference,2000.0,3.624000,4.896183,0.000000,0.000000,2.000000,4.000000,31.0
name_length_ratio,2000.0,0.866230,0.165903,0.095238,0.788534,0.928571,1.000000,1.0
address_exact,2000.0,0.089000,0.284815,0.000000,0.000000,0.000000,0.000000,1.0
address_ratio,2000.0,0.756729,0.232948,0.000000,0.642727,0.833333,0.921903,1.0


In [41]:
print("NaN values:")
print(positive_features.isna().sum())

print("\nInfinite values:")
print(np.isinf(
    positive_features.select_dtypes(include=np.number)
).sum().sum())

NaN values:
name_exact                   0
name_ratio                   0
name_token_set               0
name_token_sort              0
name_jaccard                 0
name_char_jaccard            0
name_length_difference       0
name_length_ratio            0
address_exact                0
address_ratio                0
address_token_set            0
address_token_sort           0
address_jaccard              0
address_char_jaccard         0
address_length_difference    0
address_length_ratio         0
digit_overlap                0
country_same                 0
both_exact                   0
s1_id                        0
matched_id                   0
source                       0
label                        0
dtype: int64

Infinite values:
0


In [42]:
print(positive_features.columns.tolist())
print("Number of feature columns:", len(positive_features.columns))
positive_features.head().T

['name_exact', 'name_ratio', 'name_token_set', 'name_token_sort', 'name_jaccard', 'name_char_jaccard', 'name_length_difference', 'name_length_ratio', 'address_exact', 'address_ratio', 'address_token_set', 'address_token_sort', 'address_jaccard', 'address_char_jaccard', 'address_length_difference', 'address_length_ratio', 'digit_overlap', 'country_same', 'both_exact', 's1_id', 'matched_id', 'source', 'label']
Number of feature columns: 23


,0,1,2,3,4
name_exact,0,1,1,0,0
name_ratio,0.21875,1.0,1.0,0.933333,0.948718
name_token_set,0.258065,1.0,1.0,0.933333,0.666667
name_token_sort,0.21875,1.0,1.0,0.933333,0.666667
name_jaccard,0.066667,1.0,1.0,0.6,0.428571
name_char_jaccard,0.054545,1.0,1.0,0.576923,0.731707
name_length_difference,2,0,0,1,2
name_length_ratio,0.939394,1.0,1.0,0.956522,0.95
address_exact,0,0,0,0,0
address_ratio,0.84,0.45,0.760563,0.765432,0.892308


In [43]:
feature_cols = [
    col for col in positive_features.columns
    if col not in ["s1_id", "matched_id", "source", "label"]
]

print("Features being analyzed:")
print(feature_cols)

positive_features[feature_cols].describe().T

Features being analyzed:
['name_exact', 'name_ratio', 'name_token_set', 'name_token_sort', 'name_jaccard', 'name_char_jaccard', 'name_length_difference', 'name_length_ratio', 'address_exact', 'address_ratio', 'address_token_set', 'address_token_sort', 'address_jaccard', 'address_char_jaccard', 'address_length_difference', 'address_length_ratio', 'digit_overlap', 'country_same', 'both_exact']


,count,mean,std,min,25%,50%,75%,max
name_exact,2000.0,0.221000,0.415024,0.000000,0.000000,0.000000,0.000000,1.0
name_ratio,2000.0,0.804384,0.231938,0.064516,0.740487,0.875000,0.962963,1.0
name_token_set,2000.0,0.874260,0.241243,0.066667,0.887978,1.000000,1.000000,1.0
name_token_sort,2000.0,0.806790,0.241764,0.064516,0.738963,0.892857,1.000000,1.0
name_jaccard,2000.0,0.618253,0.329361,0.000000,0.500000,0.666667,1.000000,1.0
name_char_jaccard,2000.0,0.663982,0.285356,0.000000,0.500000,0.709005,0.878788,1.0
name_length_difference,2000.0,3.624000,4.896183,0.000000,0.000000,2.000000,4.000000,31.0
name_length_ratio,2000.0,0.866230,0.165903,0.095238,0.788534,0.928571,1.000000,1.0
address_exact,2000.0,0.089000,0.284815,0.000000,0.000000,0.000000,0.000000,1.0
address_ratio,2000.0,0.756729,0.232948,0.000000,0.642727,0.833333,0.921903,1.0


In [44]:
print("NaN values:")
print(positive_features[feature_cols].isna().sum())

print("\nInfinite values:")
print(
    np.isinf(
        positive_features[feature_cols]
        .select_dtypes(include=np.number)
    ).sum().sum()
)

NaN values:
name_exact                   0
name_ratio                   0
name_token_set               0
name_token_sort              0
name_jaccard                 0
name_char_jaccard            0
name_length_difference       0
name_length_ratio            0
address_exact                0
address_ratio                0
address_token_set            0
address_token_sort           0
address_jaccard              0
address_char_jaccard         0
address_length_difference    0
address_length_ratio         0
digit_overlap                0
country_same                 0
both_exact                   0
dtype: int64

Infinite values:
0


In [45]:
# Only keep GT links for the S1s in our 2,000-pair experiment
sample_s1_ids = set(positive_sample["s1_id"])

sample_positive_links = positive_links[
    positive_links["s1_id"].isin(sample_s1_ids)
].copy()

print("Positive links for sampled S1s:", len(sample_positive_links))
print("Unique sampled S1s:", sample_positive_links["s1_id"].nunique())

Positive links for sampled S1s: 8702
Unique sampled S1s: 1998


In [46]:
positive_lookup = (
    sample_positive_links
    .groupby("s1_id")["matched_id"]
    .agg(set)
    .to_dict()
)

print("S1s in positive lookup:", len(positive_lookup))

S1s in positive lookup: 1998


In [47]:
rng = np.random.default_rng(42)

negative_rows = []

candidate_s2_ids = s2_sample["entity_id"].tolist()
candidate_s3_ids = s3_sample["entity_id"].tolist()

sample_s1_ids = positive_sample["s1_id"].unique()

for s1_id in sample_s1_ids:

    known_matches = positive_lookup.get(s1_id, set())

    attempts = 0

    while attempts < 10:
        attempts += 1

        if rng.random() < 0.5:
            matched_id = rng.choice(candidate_s2_ids)
            source = "S2"
        else:
            matched_id = rng.choice(candidate_s3_ids)
            source = "S3"

        if matched_id not in known_matches:
            negative_rows.append({
                "s1_id": s1_id,
                "matched_id": matched_id,
                "source": source,
                "label": 0
            })
            break

    if len(negative_rows) >= 2000:
        break

negative_sample = pd.DataFrame(negative_rows)

print("Negative pairs:", len(negative_sample))
print("\nSource distribution:")
print(negative_sample["source"].value_counts())

Negative pairs: 1998

Source distribution:
source
S2    1017
S3     981
Name: count, dtype: int64


In [48]:
negative_feature_rows = []

for _, pair in negative_sample.iterrows():

    s1_row = s1_lookup[pair["s1_id"]]

    if pair["source"] == "S2":
        match_row = s2_lookup[pair["matched_id"]]
    else:
        match_row = s3_lookup[pair["matched_id"]]

    features = feature_pair(s1_row, match_row)

    features["s1_id"] = pair["s1_id"]
    features["matched_id"] = pair["matched_id"]
    features["source"] = pair["source"]
    features["label"] = 0

    negative_feature_rows.append(features)

negative_features = pd.DataFrame(negative_feature_rows)

print("Negative feature rows:", len(negative_features))

Negative feature rows: 1998


In [49]:
training_sample = pd.concat(
    [positive_features, negative_features],
    ignore_index=True
)

print("Total pairs:", len(training_sample))
print("\nLabel distribution:")
print(training_sample["label"].value_counts())

print("\nSource distribution:")
print(training_sample["source"].value_counts())

Total pairs: 3998

Label distribution:
label
1    2000
0    1998
Name: count, dtype: int64

Source distribution:
source
S3    2016
S2    1982
Name: count, dtype: int64


In [50]:
negative_check = negative_sample.apply(
    lambda row: row["matched_id"] in positive_lookup.get(row["s1_id"], set()),
    axis=1
)

print("Negative pairs accidentally known as positives:", negative_check.sum())

Negative pairs accidentally known as positives: 0


In [51]:
feature_cols = [
    "name_exact",
    "name_ratio",
    "name_token_set",
    "name_token_sort",
    "name_jaccard",
    "name_char_jaccard",
    "name_length_difference",
    "name_length_ratio",
    "address_exact",
    "address_ratio",
    "address_token_set",
    "address_token_sort",
    "address_jaccard",
    "address_char_jaccard",
    "address_length_difference",
    "address_length_ratio",
    "digit_overlap",
    "country_same",
    "both_exact",
]

In [52]:
comparison = (
    training_sample
    .groupby("label")[feature_cols]
    .mean()
    .T
)

comparison.columns = [
    "negative_mean",
    "positive_mean"
]

comparison["difference"] = (
    comparison["positive_mean"]
    - comparison["negative_mean"]
)

comparison.sort_values(
    "difference",
    ascending=False
)

,negative_mean,positive_mean,difference
digit_overlap,0.004056,0.699682,0.695626
name_char_jaccard,0.019699,0.663982,0.644284
address_char_jaccard,0.017518,0.645095,0.627577
name_jaccard,0.015828,0.618253,0.602425
address_jaccard,0.009036,0.600556,0.591520
name_token_set,0.318746,0.874260,0.555514
address_token_set,0.331970,0.876016,0.544046
name_token_sort,0.311998,0.806790,0.494793
name_ratio,0.316294,0.804384,0.488090
address_token_sort,0.325504,0.813464,0.487960


In [53]:
training_sample.groupby("label")[feature_cols].median().T

label,0,1
name_exact,0.000000,0.000000
name_ratio,0.320000,0.875000
name_token_set,0.318182,1.000000
name_token_sort,0.317267,0.892857
name_jaccard,0.000000,0.666667
name_char_jaccard,0.000000,0.709005
name_length_difference,8.000000,2.000000
name_length_ratio,0.703704,0.928571
address_exact,0.000000,0.000000
address_ratio,0.317460,0.833333


In [54]:
training_sample.groupby("label")[feature_cols].quantile(
    [0.25, 0.50, 0.75]
).T

label                             0                               1            \
                               0.25       0.50       0.75      0.25      0.50   
name_exact                 0.000000   0.000000   0.000000  0.000000  0.000000   
name_ratio                 0.264151   0.320000   0.368270  0.740487  0.875000   
name_token_set             0.263158   0.318182   0.369231  0.887978  1.000000   
name_token_sort            0.262295   0.317267   0.367347  0.738963  0.892857   
name_jaccard               0.000000   0.000000   0.000000  0.500000  0.666667   
name_char_jaccard          0.000000   0.000000   0.020000  0.500000  0.709005   
name_length_difference     4.000000   8.000000  13.000000  0.000000  2.000000   
name_length_ratio          0.546188   0.703704   0.852941  0.788534  0.928571   
address_exact              0.000000   0.000000   0.000000  0.000000  0.000000   
address_ratio              0.277778   0.317460   0.358420  0.642727  0.833333   
address_token_set          0.299065   0.338028   0.380124  0.860759  0.939262   
address_token_sort         0.294118   0.333333   0.375000  0.776851  0.880000   
address_jaccard            0.000000   0.000000   0.000000  0.428571  0.633971   
address_char_jaccard       0.000000   0.009434   0.022967  0.512821  0.679711   
address_length_difference  7.000000  18.000000  36.000000  2.000000  5.000000   
address_length_ratio       0.459016   0.664729   0.837838  0.794118  0.888889   
digit_overlap              0.000000   0.000000   0.000000  0.333333  1.000000   
country_same               0.000000   1.000000   1.000000  1.000000  1.000000   
both_exact                 0.000000   0.000000   0.000000  0.000000  0.000000   

label                                 
                                0.75  
name_exact                  0.000000  
name_ratio                  0.962963  
name_token_set              1.000000  
name_token_sort             1.000000  
name_jaccard                1.000000  
name_char_jaccard           0.878788  
name_length_difference      4.000000  
name_length_ratio           1.000000  
address_exact               0.000000  
address_ratio               0.921903  
address_token_set           0.984127  
address_token_sort          0.945268  
address_jaccard             0.800000  
address_char_jaccard        0.827089  
address_length_difference  10.000000  
address_length_ratio        0.964286  
digit_overlap               1.000000  
country_same                1.000000  
both_exact                  0.000000

In [55]:
print(
    training_sample.groupby("label")[
        [
            "name_ratio",
            "name_token_set",
            "address_ratio",
            "address_token_set",
            "name_jaccard",
            "address_jaccard",
            "digit_overlap",
            "name_exact",
            "address_exact",
            "both_exact"
        ]
    ].mean()
)

       name_ratio  name_token_set  address_ratio  address_token_set  \
label                                                                 
0        0.316294        0.318746       0.310744           0.331970   
1        0.804384        0.874260       0.756729           0.876016   

       name_jaccard  address_jaccard  digit_overlap  name_exact  \
label                                                             
0          0.015828         0.009036       0.004056       0.000   
1          0.618253         0.600556       0.699682       0.221   

       address_exact  both_exact  
label                             
0              0.000      0.0000  
1              0.089      0.0145  


In [56]:
positive_features.sort_values(
    "name_ratio"
).head(20)[
    [
        "s1_id",
        "matched_id",
        "source",
        "name_ratio",
        "name_token_set",
        "address_ratio",
        "address_token_set",
        "digit_overlap"
    ]
]

,s1_id,matched_id,source,name_ratio,name_token_set,address_ratio,address_token_set,digit_overlap
374,S1-356826206,S2-377883732,S2,0.064516,0.074074,1.000000,1.000000,1.000000
1964,S1-831389333,S2-475850188,S2,0.066667,0.066667,0.595745,0.918033,1.000000
1701,S1-924694807,S3-637268063,S3,0.071429,0.071429,0.605505,0.896552,0.333333
1216,S1-570886569,S3-752881151,S3,0.078431,0.088889,0.780000,0.824742,0.500000
264,S1-56126636,S3-753466975,S3,0.080000,0.092308,0.351852,0.769231,1.000000
1013,S1-985769556,S3-496161087,S3,0.081081,0.083333,0.598131,0.646154,0.500000
844,S1-381079292,S2-951878608,S2,0.081633,0.088889,0.615385,1.000000,0.000000
827,S1-98392954,S2-559873713,S2,0.082192,0.089552,0.492958,1.000000,1.000000
768,S1-206007534,S2-423971590,S2,0.083333,0.083333,0.987448,1.000000,1.000000
788,S1-108741324,S2-228724548,S2,0.084507,0.113208,0.522523,0.909091,1.000000


In [57]:
notebook_output = ROOT / "experiments" / "feature_sample.csv"

training_sample.to_csv(
    notebook_output,
    index=False
)

print("Saved:", notebook_output)

Saved: C:\Users\manas\OneDrive\Desktop\KJSCE\amazon_ml_26\amazon-ml-challenge-2026\experiments\feature_sample.csv


In [58]:
negative_features.sort_values(
    "name_ratio",
    ascending=False
).head(20)[
    [
        "s1_id",
        "matched_id",
        "source",
        "name_ratio",
        "name_token_set",
        "address_ratio",
        "address_token_set",
        "digit_overlap"
    ]
]

,s1_id,matched_id,source,name_ratio,name_token_set,address_ratio,address_token_set,digit_overlap
1199,S1-778950463,S3-838421357,S3,0.746667,0.773333,0.358621,0.338028,0.000000
1768,S1-545268236,S2-140888254,S2,0.721311,0.721311,0.400000,0.425287,0.000000
825,S1-573179121,S2-112899334,S2,0.718750,0.718750,0.400000,0.384000,0.000000
1979,S1-619076059,S2-112899334,S2,0.718750,0.652174,0.377622,0.433566,0.000000
1742,S1-995533523,S2-840197013,S2,0.714286,0.750000,0.261538,0.243902,0.000000
1719,S1-175623547,S2-115485022,S2,0.709677,0.709677,0.350649,0.397260,0.166667
217,S1-309944195,S3-219945562,S3,0.688525,0.688525,0.324324,0.419048,0.000000
1797,S1-296123453,S2-408489007,S2,0.680851,0.638298,0.357616,0.400000,0.000000
856,S1-364147773,S2-961259779,S2,0.677966,0.769231,0.361111,0.378788,0.000000
711,S1-93726789,S2-364755434,S2,0.676471,0.652174,0.346154,0.351145,0.000000


In [59]:
negative_features.sort_values(
    "address_ratio",
    ascending=False
).head(20)[
    [
        "s1_id",
        "matched_id",
        "source",
        "name_ratio",
        "name_token_set",
        "address_ratio",
        "address_token_set",
        "digit_overlap"
    ]
]

,s1_id,matched_id,source,name_ratio,name_token_set,address_ratio,address_token_set,digit_overlap
149,S1-660321395,S2-1550441,S2,0.516129,0.516129,0.562500,0.500000,0.0
1889,S1-476405189,S3-257957769,S3,0.232558,0.325581,0.555556,0.638889,0.0
377,S1-767592924,S3-940267485,S3,0.270270,0.216216,0.545455,0.545455,0.0
263,S1-429223132,S2-527562592,S2,0.325581,0.372093,0.528302,0.566038,0.0
1091,S1-318863954,S2-297912304,S2,0.470588,0.470588,0.527778,0.614286,0.0
1827,S1-554097680,S2-520893323,S2,0.279070,0.279070,0.527778,0.472222,0.0
835,S1-173014114,S2-424806036,S2,0.291667,0.291667,0.526316,0.541667,0.0
1710,S1-276092712,S2-271680394,S2,0.227273,0.272727,0.519481,0.545455,0.0
1898,S1-37175410,S2-568104563,S2,0.357143,0.321429,0.516129,0.516129,0.0
724,S1-989989835,S2-515192548,S2,0.222222,0.222222,0.514286,0.485714,0.0


In [60]:
negative_features.sort_values(
    ["name_ratio", "address_ratio"],
    ascending=False
).head(20)[
    [
        "s1_id",
        "matched_id",
        "source",
        "name_ratio",
        "name_token_set",
        "address_ratio",
        "address_token_set",
        "digit_overlap",
        "country_same"
    ]
]

,s1_id,matched_id,source,name_ratio,name_token_set,address_ratio,address_token_set,digit_overlap,country_same
1199,S1-778950463,S3-838421357,S3,0.746667,0.773333,0.358621,0.338028,0.000000,1
1768,S1-545268236,S2-140888254,S2,0.721311,0.721311,0.400000,0.425287,0.000000,1
825,S1-573179121,S2-112899334,S2,0.718750,0.718750,0.400000,0.384000,0.000000,1
1979,S1-619076059,S2-112899334,S2,0.718750,0.652174,0.377622,0.433566,0.000000,1
1742,S1-995533523,S2-840197013,S2,0.714286,0.750000,0.261538,0.243902,0.000000,1
1719,S1-175623547,S2-115485022,S2,0.709677,0.709677,0.350649,0.397260,0.166667,1
217,S1-309944195,S3-219945562,S3,0.688525,0.688525,0.324324,0.419048,0.000000,1
1797,S1-296123453,S2-408489007,S2,0.680851,0.638298,0.357616,0.400000,0.000000,1
856,S1-364147773,S2-961259779,S2,0.677966,0.769231,0.361111,0.378788,0.000000,1
711,S1-93726789,S2-364755434,S2,0.676471,0.652174,0.346154,0.351145,0.000000,1


In [62]:
candidate_df = pd.read_csv(
    candidate_path,
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

gt_sample_df = pd.read_csv(
    gt_sample_path,
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

print("Candidate rows:", len(candidate_df))
print("GT rows:", len(gt_sample_df))

display(candidate_df.head())
display(gt_sample_df.head())

Candidate rows: 500
GT rows: 500


,source1_entity_id,candidate_entity_ids
0,S1-708796012,"S2-100115999,S2-101266972,S2-101272680,S2-1020..."
1,S1-483281264,"S2-102464027,S2-102750988,S2-105819090,S2-1063..."
2,S1-387694500,"S2-106822360,S2-107710118,S2-108701581,S2-1106..."
3,S1-973233920,"S2-100424378,S2-101521583,S2-102672815,S2-1038..."
4,S1-769172378,"S2-103273393,S2-105061954,S2-106794632,S2-1304..."


,source1_entity_id,matched_entity_ids
0,S1-708796012,"S2-994438467,S2-440628035,S3-338279058,S3-3351..."
1,S1-483281264,"S2-135813138,S2-59557708,S2-639567176,S3-99241..."
2,S1-387694500,"S2-89272121,S2-828925482,S2-960848454,S2-67434..."
3,S1-973233920,"S2-428307002,S3-110090880,S3-159133847,S3-7924..."
4,S1-769172378,"S2-977119232,S2-754328130,S2-713014713"


In [63]:
# Expand Laksh's candidate lists into one row per candidate pair

candidate_pairs = (
    candidate_df[["source1_entity_id", "candidate_entity_ids"]]
    .assign(
        matched_id=lambda df: df["candidate_entity_ids"].str.split(",")
    )
    .explode("matched_id")
    .drop(columns="candidate_entity_ids")
)

candidate_pairs["matched_id"] = candidate_pairs["matched_id"].str.strip()

print("Candidate pairs:", len(candidate_pairs))
print("Unique S1 entities:", candidate_pairs["source1_entity_id"].nunique())

display(candidate_pairs.head(10))

Candidate pairs: 945934
Unique S1 entities: 500


,source1_entity_id,matched_id
0,S1-708796012,S2-100115999
0,S1-708796012,S2-101266972
0,S1-708796012,S2-101272680
0,S1-708796012,S2-102001891
0,S1-708796012,S2-103108266
0,S1-708796012,S2-103653076
0,S1-708796012,S2-103745769
0,S1-708796012,S2-103804374
0,S1-708796012,S2-10409031
0,S1-708796012,S2-104818720


In [64]:
# Expand ground truth into one row per true match

gt_pairs = (
    gt_sample_df[["source1_entity_id", "matched_entity_ids"]]
    .assign(
        matched_id=lambda df: df["matched_entity_ids"].str.split(",")
    )
    .explode("matched_id")
    .drop(columns="matched_entity_ids")
)

gt_pairs["matched_id"] = gt_pairs["matched_id"].str.strip()

print("Ground-truth pairs:", len(gt_pairs))
print("Unique S1 entities:", gt_pairs["source1_entity_id"].nunique())

display(gt_pairs.head(10))

Ground-truth pairs: 1761
Unique S1 entities: 500


,source1_entity_id,matched_id
0,S1-708796012,S2-994438467
0,S1-708796012,S2-440628035
0,S1-708796012,S3-338279058
0,S1-708796012,S3-335174066
0,S1-708796012,S3-703951662
0,S1-708796012,S3-974934575
1,S1-483281264,S2-135813138
1,S1-483281264,S2-59557708
1,S1-483281264,S2-639567176
1,S1-483281264,S3-992417216


In [65]:
# Label every candidate pair according to whether it is a ground-truth match

gt_pair_set = set(
    zip(
        gt_pairs["source1_entity_id"],
        gt_pairs["matched_id"]
    )
)

candidate_pairs["is_match"] = [
    (s1_id, matched_id) in gt_pair_set
    for s1_id, matched_id in zip(
        candidate_pairs["source1_entity_id"],
        candidate_pairs["matched_id"]
    )
]

tp_count = int(candidate_pairs["is_match"].sum())
candidate_count = len(candidate_pairs)
gt_count = len(gt_pairs)

blocking_recall = tp_count / gt_count

print(f"Candidate pairs: {candidate_count:,}")
print(f"Ground-truth pairs: {gt_count:,}")
print(f"True matches retrieved: {tp_count:,}")
print(f"Blocking recall: {blocking_recall:.4%}")

Candidate pairs: 945,934
Ground-truth pairs: 1,761
True matches retrieved: 1,644
Blocking recall: 93.3560%


In [66]:
candidate_pairs["source"] = candidate_pairs["matched_id"].str[:2]

print("Candidate source distribution:")
display(candidate_pairs["source"].value_counts())

print("\nTrue-match source distribution:")
display(
    candidate_pairs.loc[candidate_pairs["is_match"], "source"]
    .value_counts()
)

Candidate source distribution:


source
S3    489850
S2    456084
Name: count, dtype: int64


True-match source distribution:


source
S3    841
S2    803
Name: count, dtype: int64

In [67]:
print("Candidate pair labels:")
display(candidate_pairs["is_match"].value_counts())

print("\nPositive rate:")
print(f"{candidate_pairs['is_match'].mean():.4%}")

Candidate pair labels:


is_match
False    944290
True       1644
Name: count, dtype: int64


Positive rate:
0.1738%


In [68]:
# IDs needed for feature extraction

needed_s1_ids = set(candidate_pairs["source1_entity_id"])

needed_s2_ids = set(
    candidate_pairs.loc[
        candidate_pairs["source"] == "S2", "matched_id"
    ]
)

needed_s3_ids = set(
    candidate_pairs.loc[
        candidate_pairs["source"] == "S3", "matched_id"
    ]
)

print("S1 records needed:", len(needed_s1_ids))
print("S2 records needed:", len(needed_s2_ids))
print("S3 records needed:", len(needed_s3_ids))

S1 records needed: 500
S2 records needed: 285561
S3 records needed: 305036


In [69]:
def load_rows_by_ids(path, needed_ids, chunksize=200_000):
    pieces = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize,
    ):
        matched = chunk[chunk["entity_id"].isin(needed_ids)]

        if not matched.empty:
            pieces.append(matched)

        if sum(len(x) for x in pieces) >= len(needed_ids):
            break

    if not pieces:
        return pd.DataFrame()

    return pd.concat(pieces, ignore_index=True)

In [70]:
s1_candidates = load_rows_by_ids(
    DATA_DIR / "train_source1.tsv",
    needed_s1_ids,
)

s2_candidates = load_rows_by_ids(
    DATA_DIR / "train_source2.tsv",
    needed_s2_ids,
)

s3_candidates = load_rows_by_ids(
    DATA_DIR / "train_source3.tsv",
    needed_s3_ids,
)

print("Loaded S1:", len(s1_candidates))
print("Loaded S2:", len(s2_candidates))
print("Loaded S3:", len(s3_candidates))

Loaded S1: 500
Loaded S2: 285561
Loaded S3: 305036


In [71]:
# Prepare normalized fields

def prepare_source(df):
    df = df.copy()
    df["name_norm"] = df["business_name"].map(normalize_name)
    df["address_norm"] = df["business_address"].map(normalize_address)
    return df


s1_lookup = (
    prepare_source(s1_candidates)
    .set_index("entity_id")
    .to_dict("index")
)

s2_lookup = (
    prepare_source(s2_candidates)
    .set_index("entity_id")
    .to_dict("index")
)

s3_lookup = (
    prepare_source(s3_candidates)
    .set_index("entity_id")
    .to_dict("index")
)

print("S1 lookup:", len(s1_lookup))
print("S2 lookup:", len(s2_lookup))
print("S3 lookup:", len(s3_lookup))

S1 lookup: 500
S2 lookup: 285561
S3 lookup: 305036


In [72]:
# Check that every candidate has a corresponding source record

missing_s1 = sum(
    s1_id not in s1_lookup
    for s1_id in candidate_pairs["source1_entity_id"]
)

missing_s2 = sum(
    matched_id not in s2_lookup
    for matched_id in candidate_pairs.loc[
        candidate_pairs["source"] == "S2", "matched_id"
    ]
)

missing_s3 = sum(
    matched_id not in s3_lookup
    for matched_id in candidate_pairs.loc[
        candidate_pairs["source"] == "S3", "matched_id"
    ]
)

print("Missing S1 records:", missing_s1)
print("Missing S2 records:", missing_s2)
print("Missing S3 records:", missing_s3)

Missing S1 records: 0
Missing S2 records: 0
Missing S3 records: 0


In [73]:
# Extract features for candidate pairs in batches

feature_columns = [
    "name_exact",
    "name_ratio",
    "name_token_set",
    "name_token_sort",
    "name_jaccard",
    "name_char_jaccard",
    "name_length_difference",
    "name_length_ratio",
    "address_exact",
    "address_ratio",
    "address_token_set",
    "address_token_sort",
    "address_jaccard",
    "address_char_jaccard",
    "address_length_difference",
    "address_length_ratio",
    "digit_overlap",
    "country_same",
    "both_exact",
]

def extract_candidate_features(candidate_df, batch_size=25_000):
    rows = []

    total = len(candidate_df)

    for start in range(0, total, batch_size):
        batch = candidate_df.iloc[start:start + batch_size]

        for _, pair in batch.iterrows():
            s1_id = pair["source1_entity_id"]
            matched_id = pair["matched_id"]
            source = pair["source"]

            s1_row = s1_lookup[s1_id]

            if source == "S2":
                match_row = s2_lookup[matched_id]
            else:
                match_row = s3_lookup[matched_id]

            features = feature_pair(s1_row, match_row)

            features["s1_id"] = s1_id
            features["matched_id"] = matched_id
            features["source"] = source

            rows.append(features)

        print(
            f"Processed {min(start + batch_size, total):,} / {total:,}"
        )

    return pd.DataFrame(rows)


candidate_features = extract_candidate_features(candidate_pairs)

print("Feature rows:", len(candidate_features))
print("Feature columns:", len(candidate_features.columns))

Processed 25,000 / 945,934
Processed 50,000 / 945,934
Processed 75,000 / 945,934
Processed 100,000 / 945,934
Processed 125,000 / 945,934
Processed 150,000 / 945,934
Processed 175,000 / 945,934
Processed 200,000 / 945,934
Processed 225,000 / 945,934
Processed 250,000 / 945,934
Processed 275,000 / 945,934
Processed 300,000 / 945,934
Processed 325,000 / 945,934
Processed 350,000 / 945,934
Processed 375,000 / 945,934
Processed 400,000 / 945,934
Processed 425,000 / 945,934
Processed 450,000 / 945,934
Processed 475,000 / 945,934
Processed 500,000 / 945,934
Processed 525,000 / 945,934
Processed 550,000 / 945,934
Processed 575,000 / 945,934
Processed 600,000 / 945,934
Processed 625,000 / 945,934
Processed 650,000 / 945,934
Processed 675,000 / 945,934
Processed 700,000 / 945,934
Processed 725,000 / 945,934
Processed 750,000 / 945,934
Processed 775,000 / 945,934
Processed 800,000 / 945,934
Processed 825,000 / 945,934
Processed 850,000 / 945,934
Processed 875,000 / 945,934
Processed 900,000 / 945

In [74]:
# Attach candidate labels to the feature matrix

candidate_features["is_match"] = candidate_pairs["is_match"].to_numpy()

print(candidate_features["is_match"].value_counts())
print(f"Positive rate: {candidate_features['is_match'].mean():.4%}")

is_match
False    944290
True       1644
Name: count, dtype: int64
Positive rate: 0.1738%


In [75]:
feature_output_path = ROOT / "experiments" / "candidate_features_sample.csv"

candidate_features.to_csv(
    feature_output_path,
    index=False,
)

print("Saved:", feature_output_path)
print("Rows:", len(candidate_features))

Saved: C:\Users\manas\OneDrive\Desktop\KJSCE\amazon_ml_26\amazon-ml-challenge-2026\experiments\candidate_features_sample.csv
Rows: 945934


In [76]:
print(
    f"File size: {feature_output_path.stat().st_size / (1024**2):.2f} MB"
)

File size: 219.42 MB


In [77]:
# True vs false candidate feature means

analysis_columns = [
    col for col in feature_columns
    if col in candidate_features.columns
]

feature_summary = (
    candidate_features
    .groupby("is_match")[analysis_columns]
    .mean()
    .T
    .rename(columns={
        False: "false_mean",
        True: "true_mean",
    })
)

feature_summary["difference"] = (
    feature_summary["true_mean"]
    - feature_summary["false_mean"]
)

feature_summary["abs_difference"] = (
    feature_summary["difference"].abs()
)

feature_summary = feature_summary.sort_values(
    "abs_difference",
    ascending=False,
)

display(feature_summary)

is_match,false_mean,true_mean,difference,abs_difference
address_length_difference,21.762550,9.406326,-12.356224,12.356224
name_length_difference,9.261456,3.313869,-5.947587,5.947587
name_char_jaccard,0.091721,0.681463,0.589743,0.589743
address_char_jaccard,0.053451,0.634335,0.580884,0.580884
name_jaccard,0.093122,0.653068,0.559946,0.559946
address_jaccard,0.062918,0.585839,0.522921,0.522921
name_token_set,0.410280,0.896513,0.486233,0.486233
address_token_set,0.398732,0.870758,0.472025,0.472025
name_token_sort,0.377107,0.831583,0.454476,0.454476
name_ratio,0.384115,0.821637,0.437522,0.437522


In [78]:
exact_features = [
    "name_exact",
    "address_exact",
    "both_exact",
]

for col in exact_features:
    print(f"\n{col}")
    print(
        candidate_features
        .groupby("is_match")[col]
        .mean()
    )


name_exact
is_match
False    0.001085
True     0.231752
Name: name_exact, dtype: float64

address_exact
is_match
False    0.000013
True     0.069343
Name: address_exact, dtype: float64

both_exact
is_match
False    0.000000
True     0.012774
Name: both_exact, dtype: float64


In [79]:
country_summary = pd.crosstab(
    candidate_features["country_same"],
    candidate_features["is_match"],
    normalize="columns",
)

display(country_summary)

is_match,False,True
country_same,,
0,0.319785,0.0
1,0.680215,1.0


In [80]:
display(
    candidate_features
    .groupby("country_same")["is_match"]
    .agg(
        candidate_pairs="size",
        true_matches="sum",
        precision="mean",
    )
)

,candidate_pairs,true_matches,precision
country_same,,,
0,301970,0,0.000000
1,643964,1644,0.002553


In [81]:
digit_bins = pd.cut(
    candidate_features["digit_overlap"],
    bins=[-0.001, 0, 0.25, 0.5, 0.75, 1.0],
)

digit_analysis = (
    candidate_features
    .groupby(digit_bins, observed=False)["is_match"]
    .agg(
        candidate_pairs="size",
        true_matches="sum",
        precision="mean",
    )
)

display(digit_analysis)

,candidate_pairs,true_matches,precision
digit_overlap,,,
"(-0.001, 0.0]",294004,318,0.001082
"(0.0, 0.25]",153840,8,0.000052
"(0.25, 0.5]",358884,241,0.000672
"(0.5, 0.75]",2757,45,0.016322
"(0.75, 1.0]",136449,1032,0.007563


In [82]:
for col in [
    "name_ratio",
    "name_token_set",
    "name_jaccard",
    "name_char_jaccard",
    "address_ratio",
    "address_token_set",
    "address_jaccard",
    "address_char_jaccard",
]:
    print(f"\n===== {col} =====")

    bins = pd.cut(
        candidate_features[col],
        bins=[-0.001, 0.25, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    )

    result = (
        candidate_features
        .groupby(bins, observed=False)["is_match"]
        .agg(
            candidates="size",
            true_matches="sum",
            precision="mean",
        )
    )

    display(result)


===== name_ratio =====


,candidates,true_matches,precision
name_ratio,,,
"(-0.001, 0.25]",167486,91,0.000543
"(0.25, 0.5]",562056,26,0.000046
"(0.5, 0.6]",81735,39,0.000477
"(0.6, 0.7]",77948,142,0.001822
"(0.7, 0.8]",42475,243,0.005721
"(0.8, 0.9]",10484,371,0.035387
"(0.9, 1.0]",3750,732,0.195200



===== name_token_set =====


,candidates,true_matches,precision
name_token_set,,,
"(-0.001, 0.25]",169956,90,0.000530
"(0.25, 0.5]",529518,23,0.000043
"(0.5, 0.6]",49275,28,0.000568
"(0.6, 0.7]",64067,28,0.000437
"(0.7, 0.8]",66757,60,0.000899
"(0.8, 0.9]",46839,186,0.003971
"(0.9, 1.0]",19522,1229,0.062955



===== name_jaccard =====


...

,candidates,true_matches,precision
address_token_set,,,
"(-0.001, 0.25]",16068,69,0.004294
"(0.25, 0.5]",836560,6,0.000007
"(0.5, 0.6]",67351,9,0.000134
"(0.6, 0.7]",16582,47,0.002834
"(0.7, 0.8]",4960,129,0.026008
"(0.8, 0.9]",2674,397,0.148467
"(0.9, 1.0]",1739,987,0.567568



===== address_jaccard =====


,candidates,true_matches,precision
address_jaccard,,,
"(-0.001, 0.25]",937341,173,0.000185
"(0.25, 0.5]",7225,512,0.070865
"(0.5, 0.6]",350,171,0.488571
"(0.6, 0.7]",370,243,0.656757
"(0.7, 0.8]",275,205,0.745455
"(0.8, 0.9]",122,106,0.868852
"(0.9, 1.0]",251,234,0.932271



===== address_char_jaccard =====


,candidates,true_matches,precision
address_char_jaccard,,,
"(-0.001, 0.25]",931510,133,0.000143
"(0.25, 0.5]",12027,293,0.024362
"(0.5, 0.6]",879,224,0.254835
"(0.6, 0.7]",597,282,0.472362
"(0.7, 0.8]",418,278,0.665072
"(0.8, 0.9]",285,235,0.824561
"(0.9, 1.0]",218,199,0.912844


In [83]:
hard_true = (
    candidate_features[
        candidate_features["is_match"]
    ]
    .sort_values(
        ["name_ratio", "address_ratio"]
    )
)

display(
    hard_true[
        [
            "s1_id",
            "matched_id",
            "source",
            "name_ratio",
            "name_token_set",
            "name_jaccard",
            "name_char_jaccard",
            "address_ratio",
            "address_token_set",
            "address_jaccard",
            "digit_overlap",
            "country_same",
        ]
    ].head(30)
)

,s1_id,matched_id,source,name_ratio,name_token_set,name_jaccard,name_char_jaccard,address_ratio,address_token_set,address_jaccard,digit_overlap,country_same
880787,S1-348739044,S2-149870688,S2,0.057143,0.060606,0.0,0.0,0.985507,1.000000,0.916667,1.000000,1
818399,S1-866107291,S2-637744180,S2,0.062500,0.066667,0.0,0.0,0.600000,0.769231,0.333333,0.500000,1
818269,S1-866107291,S2-547351777,S2,0.062500,0.066667,0.0,0.0,0.727273,0.923077,0.666667,0.500000,1
819141,S1-866107291,S3-262787266,S3,0.062500,0.066667,0.0,0.0,0.899083,0.950000,0.545455,0.000000,1
375674,S1-679363660,S3-577079683,S3,0.066667,0.071429,0.0,0.0,0.777143,0.865248,0.571429,1.000000,1
375614,S1-679363660,S2-93630949,S2,0.066667,0.071429,0.0,0.0,0.851852,0.918919,0.684211,1.000000,1
375412,S1-679363660,S2-622558605,S2,0.066667,0.071429,0.0,0.0,1.000000,1.000000,1.000000,1.000000,1
495646,S1-886602086,S3-260615697,S3,0.070175,0.070175,0.0,0.0,0.903226,0.944882,0.785714,0.000000,1
495369,S1-886602086,S2-602904740,S2,0.070175,0.070175,0.0,0.0,0.907975,0.952381,0.733333,0.000000,1
380895,S1-328864149,S2-542245819,S2,0.075472,0.078431,0.0,0.0,0.615385,0.914286,0.642857,1.000000,1


In [84]:
hard_false = (
    candidate_features[
        ~candidate_features["is_match"]
    ]
    .sort_values(
        ["name_ratio", "address_ratio"],
        ascending=False,
    )
)

display(
    hard_false[
        [
            "s1_id",
            "matched_id",
            "source",
            "name_ratio",
            "name_token_set",
            "name_jaccard",
            "address_ratio",
            "address_token_set",
            "address_jaccard",
            "digit_overlap",
            "country_same",
        ]
    ].head(30)
)

,s1_id,matched_id,source,name_ratio,name_token_set,name_jaccard,address_ratio,address_token_set,address_jaccard,digit_overlap,country_same
387681,S1-533802013,S3-897485852,S3,1.0,1.0,1.0,0.622951,0.524590,0.111111,0.0,1
107920,S1-185055467,S3-451890135,S3,1.0,1.0,1.0,0.611940,0.631579,0.187500,0.0,1
864399,S1-466989811,S2-210168096,S2,1.0,1.0,1.0,0.562500,0.531250,0.111111,0.0,1
668462,S1-912818025,S2-973958391,S2,1.0,1.0,1.0,0.543046,0.553191,0.117647,0.0,1
119672,S1-212363226,S2-819842986,S2,1.0,1.0,1.0,0.535211,0.507042,0.090909,0.0,1
376581,S1-530648602,S2-348032183,S2,1.0,1.0,1.0,0.528302,0.528302,0.111111,0.0,1
377704,S1-530648602,S3-832032867,S3,1.0,1.0,1.0,0.524590,0.491803,0.111111,0.0,1
384896,S1-533802013,S2-353394453,S2,1.0,1.0,1.0,0.517241,0.482759,0.111111,0.0,1
121391,S1-212363226,S3-763855029,S3,1.0,1.0,1.0,0.514286,0.476190,0.090909,0.0,1
830390,S1-970660281,S2-11406347,S2,1.0,1.0,1.0,0.510638,0.510638,0.066667,0.0,1


In [85]:
candidate_pair_set = set(
    zip(
        candidate_pairs["source1_entity_id"],
        candidate_pairs["matched_id"],
    )
)

gt_pairs["in_candidate_set"] = [
    (s1_id, matched_id) in candidate_pair_set
    for s1_id, matched_id in zip(
        gt_pairs["source1_entity_id"],
        gt_pairs["matched_id"],
    )
]

blocking_misses = gt_pairs[
    ~gt_pairs["in_candidate_set"]
].copy()

print("Blocking misses:", len(blocking_misses))

display(blocking_misses.head(30))

Blocking misses: 117


,source1_entity_id,matched_id,in_candidate_set
2,S1-387694500,S2-89272121,False
2,S1-387694500,S2-828925482,False
2,S1-387694500,S2-674345300,False
2,S1-387694500,S3-789542813,False
2,S1-387694500,S3-9126403,False
2,S1-387694500,S3-904486811,False
2,S1-387694500,S3-77197957,False
5,S1-881535873,S2-642463929,False
6,S1-436985273,S3-150259143,False
9,S1-338441694,,False


In [86]:
# Inspect the actual raw records behind difficult true positives

hard_true_ids = hard_true.head(10)[
    ["s1_id", "matched_id", "source"]
].to_dict("records")

for pair in hard_true_ids:
    s1 = s1_lookup[pair["s1_id"]]

    if pair["source"] == "S2":
        match = s2_lookup[pair["matched_id"]]
    else:
        match = s3_lookup[pair["matched_id"]]

    print("=" * 100)
    print("S1:", pair["s1_id"])
    print("MATCH:", pair["matched_id"], pair["source"])
    print()
    print("S1 name:     ", s1["business_name"])
    print("MATCH name:  ", match["business_name"])
    print()
    print("S1 address:  ", s1["business_address"])
    print("MATCH address:", match["business_address"])
    print()
    print("S1 country:  ", s1["country"])
    print("MATCH country:", match["country"])

S1: S1-348739044
MATCH: S2-149870688 S2

S1 name:      Dynamic Foundation
MATCH name:   டைனமிக் ஃபவுண்டேஷன்

S1 address:   No. 79, Mc Nichols Road, Chetpet, Egmore Nungambakkam, Chennai, Tamil Nadu
MATCH address: H.NO 79, MC NICHOLS ROAD, CHETPET, EGMORE NUNGAMBAKKAM, CHENNAI, Tamil Nadu

S1 country:   India
MATCH country: India
S1: S1-866107291
MATCH: S2-637744180 S2

S1 name:      Sree Technologies
MATCH name:   श्री टेक्नोलॉजीज

S1 address:   148, Katra Mashroo, Darbia Kalan, Delhi, North Delhi, Delhi
MATCH address: PLOT 581 148, DELHI, NORTH DELHI, Delhi

S1 country:   India
MATCH country: India
S1: S1-866107291
MATCH: S2-547351777 S2

S1 name:      Sree Technologies
MATCH name:   श्री टेक्नोलॉजीज

S1 address:   148, Katra Mashroo, Darbia Kalan, Delhi, North Delhi, Delhi
MATCH address: PLTO 581 148, KATRA MASHROO, DARBIA KALAN, DELHI

S1 country:   India
MATCH country: India
S1: S1-866107291
MATCH: S3-262787266 S3

S1 name:      Sree Technologies
MATCH name:   श्री टेक्नोलॉजीज

S1 

In [87]:
hard_false_ids = hard_false.head(10)[
    ["s1_id", "matched_id", "source"]
].to_dict("records")

for pair in hard_false_ids:
    s1 = s1_lookup[pair["s1_id"]]

    if pair["source"] == "S2":
        match = s2_lookup[pair["matched_id"]]
    else:
        match = s3_lookup[pair["matched_id"]]

    print("=" * 100)
    print("S1:", pair["s1_id"])
    print("FALSE CANDIDATE:", pair["matched_id"], pair["source"])
    print()
    print("S1 name:     ", s1["business_name"])
    print("MATCH name:  ", match["business_name"])
    print()
    print("S1 address:  ", s1["business_address"])
    print("MATCH address:", match["business_address"])
    print()
    print("S1 country:  ", s1["country"])
    print("MATCH country:", match["country"])

S1: S1-533802013
FALSE CANDIDATE: S3-897485852 S3

S1 name:      American Fund
MATCH name:   AMERICAN FUND

S1 address:   135 Whitaker Street, Portland, OR
MATCH address: 726 Lambert St, Portland, Oregon

S1 country:   US
MATCH country: US
S1: S1-185055467
FALSE CANDIDATE: S3-451890135 S3

S1 name:      Bangalore South Labs Private Limited
MATCH name:   bangalore south labs private limited

S1 address:   No. 19, 4Th C, Koramangala Industrial Area, 5Th Block, Bangalore South, Bangalore, Karnataka
MATCH address: Sy No.191 Bidarhalli, Bangalore South, Bangalore, KA

S1 country:   India
MATCH country: India
S1: S1-466989811
FALSE CANDIDATE: S2-210168096 S2

S1 name:      Zenith, LLC
MATCH name:   Zenith LLC

S1 address:   7506 Glenvista Street, Houston, TX
MATCH address: 0730 ELEVENTH STREET, MOORHEAD, MN

S1 country:   US
MATCH country: US
S1: S1-912818025
FALSE CANDIDATE: S2-973958391 S2

S1 name:      Quick & Co
MATCH name:   Quick + (Co)

S1 address:   : #403, Patels Residency, Nanjund

In [89]:
# IDs of true matches that blocking failed to retrieve

missed_s2_ids = set(
    blocking_misses.loc[
        blocking_misses["matched_id"].str.startswith("S2-"),
        "matched_id"
    ]
)

missed_s3_ids = set(
    blocking_misses.loc[
        blocking_misses["matched_id"].str.startswith("S3-"),
        "matched_id"
    ]
)

print("Missed S2 IDs:", len(missed_s2_ids))
print("Missed S3 IDs:", len(missed_s3_ids))

Missed S2 IDs: 52
Missed S3 IDs: 37


In [90]:
missed_s2_records = load_rows_by_ids(
    DATA_DIR / "train_source2.tsv",
    missed_s2_ids,
)

missed_s3_records = load_rows_by_ids(
    DATA_DIR / "train_source3.tsv",
    missed_s3_ids,
)

print("Loaded missed S2 records:", len(missed_s2_records))
print("Loaded missed S3 records:", len(missed_s3_records))

Loaded missed S2 records: 52
Loaded missed S3 records: 37


In [91]:
missed_s2_lookup = (
    prepare_source(missed_s2_records)
    .set_index("entity_id")
    .to_dict("index")
)

missed_s3_lookup = (
    prepare_source(missed_s3_records)
    .set_index("entity_id")
    .to_dict("index")
)

In [92]:
miss_ids = blocking_misses.head(15)[
    ["source1_entity_id", "matched_id"]
].to_dict("records")

for pair in miss_ids:
    s1 = s1_lookup[pair["source1_entity_id"]]
    matched_id = pair["matched_id"]

    if matched_id.startswith("S2-"):
        match = missed_s2_lookup.get(matched_id)
    else:
        match = missed_s3_lookup.get(matched_id)

    print("=" * 100)
    print("S1:", pair["source1_entity_id"])
    print("MISSED TRUE MATCH:", matched_id)
    print()
    print("S1 name:     ", s1["business_name"])
    print("MATCH name:  ", match["business_name"])
    print()
    print("S1 address:  ", s1["business_address"])
    print("MATCH address:", match["business_address"])
    print()
    print("S1 country:  ", s1["country"])
    print("MATCH country:", match["country"])

S1: S1-387694500
MISSED TRUE MATCH: S2-89272121

S1 name:      Bombay Power Private Limited
MATCH name:   ಬಾಂಬೆ ಪವರ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್

S1 address:   No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma Soft Tech Park, Whitefield Main Road, Ramagon, Danahalli, Bangalore, Karnataka
MATCH address: NO: 032/2, 2ND FLOOR, MCC TOWERS, OPP TO SIGMA SOFT TECH PARK, WHITEFIELD MAIN ROAD, RAMAGON, DANAHALLI, BANGALORE, Karnataka

S1 country:   India
MATCH country: India
S1: S1-387694500
MISSED TRUE MATCH: S2-828925482

S1 name:      Bombay Power Private Limited
MATCH name:   Mr Bombay Poer Private Limited

S1 address:   No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma Soft Tech Park, Whitefield Main Road, Ramagon, Danahalli, Bangalore, Karnataka
MATCH address: NO: 032/2, 2ND FLOOR, MCC TOWERS, OPP TO SIGMA SOT TECH PARK, WHITEFIELD MAIN ROAD, RAMAGON, DANAHALLI, BANGALORE, ಕರ್ನಾಟಕ

S1 country:   India
MATCH country: India
S1: S1-387694500
MISSED TRUE MATCH: S2-674345300

S1 name:      Bombay Power Private

TypeError: 'NoneType' object is not subscriptable

In [93]:
def load_rows_by_ids(path, needed_ids, chunksize=200_000):
    needed_ids = set(needed_ids)
    pieces = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize,
    ):
        matched = chunk[chunk["entity_id"].isin(needed_ids)]

        if not matched.empty:
            pieces.append(matched)

    if not pieces:
        return pd.DataFrame()

    result = pd.concat(pieces, ignore_index=True)

    # Keep only requested IDs and remove accidental duplicates
    result = result[result["entity_id"].isin(needed_ids)]
    result = result.drop_duplicates("entity_id")

    return result

In [94]:
missed_s2_records = load_rows_by_ids(
    DATA_DIR / "train_source2.tsv",
    missed_s2_ids,
)

missed_s3_records = load_rows_by_ids(
    DATA_DIR / "train_source3.tsv",
    missed_s3_ids,
)

print("Requested S2:", len(missed_s2_ids))
print("Loaded S2:", len(missed_s2_records))

print("Requested S3:", len(missed_s3_ids))
print("Loaded S3:", len(missed_s3_records))

Requested S2: 52
Loaded S2: 52
Requested S3: 37
Loaded S3: 37


In [95]:
missed_s2_lookup = (
    prepare_source(missed_s2_records)
    .set_index("entity_id")
    .to_dict("index")
)

missed_s3_lookup = (
    prepare_source(missed_s3_records)
    .set_index("entity_id")
    .to_dict("index")
)

In [96]:
missing_from_lookup = []

for pair in blocking_misses[["source1_entity_id", "matched_id"]].to_dict("records"):
    if pair["matched_id"].startswith("S2-"):
        exists = pair["matched_id"] in missed_s2_lookup
    else:
        exists = pair["matched_id"] in missed_s3_lookup

    if not exists:
        missing_from_lookup.append(pair["matched_id"])

print("Missed GT IDs not found in source lookup:", len(missing_from_lookup))
print(missing_from_lookup[:20])

Missed GT IDs not found in source lookup: 28
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']


In [97]:
# Keep only actual ground-truth links
blocking_misses = blocking_misses[
    blocking_misses["matched_id"].ne("")
].copy()

print("Actual blocking misses:", len(blocking_misses))
print(
    "Empty matched IDs remaining:",
    blocking_misses["matched_id"].eq("").sum()
)

Actual blocking misses: 89
Empty matched IDs remaining: 0


In [98]:
missed_s2_ids = set(
    blocking_misses.loc[
        blocking_misses["matched_id"].str.startswith("S2-"),
        "matched_id"
    ]
)

missed_s3_ids = set(
    blocking_misses.loc[
        blocking_misses["matched_id"].str.startswith("S3-"),
        "matched_id"
    ]
)

print("Missed S2 IDs:", len(missed_s2_ids))
print("Missed S3 IDs:", len(missed_s3_ids))
print("Total:", len(missed_s2_ids) + len(missed_s3_ids))

Missed S2 IDs: 52
Missed S3 IDs: 37
Total: 89


In [99]:
print("Blocking miss links:", len(blocking_misses))
print("Unique missed IDs:", blocking_misses["matched_id"].nunique())
print("Unique S1s with misses:", blocking_misses["source1_entity_id"].nunique())

print("\nSource distribution:")
print(blocking_misses["matched_id"].str[:2].value_counts())

Blocking miss links: 89
Unique missed IDs: 89
Unique S1s with misses: 57

Source distribution:
matched_id
S2    52
S3    37
Name: count, dtype: int64


In [100]:
missing_from_lookup = []

for matched_id in blocking_misses["matched_id"].unique():

    if matched_id.startswith("S2-"):
        exists = matched_id in missed_s2_lookup
    elif matched_id.startswith("S3-"):
        exists = matched_id in missed_s3_lookup
    else:
        exists = False

    if not exists:
        missing_from_lookup.append(matched_id)

print("Unique missed IDs not found:", len(missing_from_lookup))
print(missing_from_lookup)

Unique missed IDs not found: 0
[]


In [101]:
miss_feature_rows = []

for _, pair in blocking_misses.iterrows():

    s1 = s1_lookup[pair["source1_entity_id"]]

    if pair["matched_id"].startswith("S2-"):
        match = missed_s2_lookup[pair["matched_id"]]
        source = "S2"
    else:
        match = missed_s3_lookup[pair["matched_id"]]
        source = "S3"

    features = feature_pair(s1, match)

    features["s1_id"] = pair["source1_entity_id"]
    features["matched_id"] = pair["matched_id"]
    features["source"] = source

    miss_feature_rows.append(features)

blocking_miss_features = pd.DataFrame(miss_feature_rows)

print("Blocking miss feature rows:", len(blocking_miss_features))

Blocking miss feature rows: 89


In [102]:
blocking_miss_features[
    [
        "name_ratio",
        "name_token_set",
        "name_jaccard",
        "name_char_jaccard",
        "address_ratio",
        "address_token_set",
        "address_jaccard",
        "address_char_jaccard",
        "digit_overlap",
        "country_same",
    ]
].describe()

,name_ratio,name_token_set,name_jaccard,name_char_jaccard,address_ratio,address_token_set,address_jaccard,address_char_jaccard,digit_overlap,country_same
count,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000,89.0
mean,0.428111,0.432349,0.106429,0.203838,0.629635,0.847314,0.532756,0.541458,0.441386,1.0
std,0.318639,0.318759,0.190523,0.232783,0.245112,0.240679,0.286248,0.278933,0.423319,0.0
min,0.081633,0.093023,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
25%,0.109091,0.122449,0.000000,0.000000,0.480000,0.810811,0.333333,0.310345,0.000000,1.0
50%,0.333333,0.307692,0.000000,0.000000,0.654545,0.935673,0.533333,0.563380,0.333333,1.0
75%,0.744186,0.755556,0.200000,0.384615,0.821429,0.995671,0.800000,0.763636,1.000000,1.0
max,0.969697,0.969697,0.600000,0.823529,1.000000,1.000000,1.000000,1.000000,1.000000,1.0


In [103]:
feature_cols = [
    "name_ratio",
    "name_token_set",
    "name_jaccard",
    "name_char_jaccard",
    "address_ratio",
    "address_token_set",
    "address_jaccard",
    "address_char_jaccard",
    "digit_overlap",
    "country_same",
]

retrieved_true = candidate_features[
    candidate_features["is_match"] == True
].copy()

comparison = pd.DataFrame({
    "retrieved_true_mean": retrieved_true[feature_cols].mean(),
    "blocking_miss_mean": blocking_miss_features[feature_cols].mean(),
})

comparison["miss_minus_retrieved"] = (
    comparison["blocking_miss_mean"]
    - comparison["retrieved_true_mean"]
)

comparison.sort_values("miss_minus_retrieved")

,retrieved_true_mean,blocking_miss_mean,miss_minus_retrieved
name_jaccard,0.653068,0.106429,-0.546639
name_char_jaccard,0.681463,0.203838,-0.477625
name_token_set,0.896513,0.432349,-0.464163
name_ratio,0.821637,0.428111,-0.393526
digit_overlap,0.712987,0.441386,-0.271601
address_ratio,0.757127,0.629635,-0.127492
address_char_jaccard,0.634335,0.541458,-0.092877
address_jaccard,0.585839,0.532756,-0.053083
address_token_set,0.870758,0.847314,-0.023444
country_same,1.000000,1.000000,0.000000


In [105]:
# Convert the existing S1 lookup into a DataFrame
s1_records = pd.DataFrame.from_dict(
    s1_lookup,
    orient="index"
).reset_index()

s1_records = s1_records.rename(columns={"index": "entity_id"})

print(s1_records.columns.tolist())
print("S1 records:", len(s1_records))

['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'address_norm']
S1 records: 500


In [106]:
miss_analysis = blocking_miss_features.copy()

miss_analysis = miss_analysis.merge(
    s1_records[
        ["entity_id", "business_name", "business_address", "country"]
    ],
    left_on="s1_id",
    right_on="entity_id",
    how="left"
)

miss_analysis = miss_analysis.rename(columns={
    "business_name": "s1_name",
    "business_address": "s1_address",
    "country": "s1_country",
})

miss_analysis.drop(columns=["entity_id"], inplace=True)

In [107]:
source_records = pd.concat(
    [missed_s2_records, missed_s3_records],
    ignore_index=True
)

source_records = source_records[
    ["entity_id", "business_name", "business_address", "country"]
].drop_duplicates("entity_id")

miss_analysis = miss_analysis.merge(
    source_records,
    left_on="matched_id",
    right_on="entity_id",
    how="left"
)

miss_analysis = miss_analysis.rename(columns={
    "business_name": "matched_name",
    "business_address": "matched_address",
    "country": "matched_country",
})

miss_analysis.drop(columns=["entity_id"], inplace=True)

In [108]:
print("Rows:", len(miss_analysis))

print(
    "Missing S1 names:",
    miss_analysis["s1_name"].isna().sum()
)

print(
    "Missing matched names:",
    miss_analysis["matched_name"].isna().sum()
)

Rows: 89
Missing S1 names: 0
Missing matched names: 0


In [109]:
miss_analysis[
    [
        "s1_id",
        "matched_id",
        "s1_name",
        "matched_name",
        "s1_address",
        "matched_address",
        "name_ratio",
        "name_token_set",
        "name_jaccard",
        "address_ratio",
        "address_token_set",
        "address_jaccard",
        "digit_overlap",
    ]
].head(30)

,s1_id,matched_id,s1_name,matched_name,s1_address,matched_address,name_ratio,name_token_set,name_jaccard,address_ratio,address_token_set,address_jaccard,digit_overlap
0,S1-387694500,S2-89272121,Bombay Power Private Limited,ಬಾಂಬೆ ಪವರ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma ...","NO: 032/2, 2ND FLOOR, MCC TOWERS, OPP TO SIGMA...",0.117647,0.127660,0.000000,0.995671,0.995671,0.904762,0.333333
1,S1-387694500,S2-828925482,Bombay Power Private Limited,Mr Bombay Poer Private Limited,"No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma ...","NO: 032/2, 2ND FLOOR, MCC TOWERS, OPP TO SIGMA...",0.931034,0.931034,0.500000,0.921053,0.923810,0.680000,0.333333
2,S1-387694500,S2-674345300,Bombay Power Private Limited,ಬಾಂಬೆ ಪವರ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma ...","BANGALORE, 2ND FLOOR, MCC TOWERS, OPP TO SIGMA...",0.117647,0.127660,0.000000,0.865801,0.995671,0.904762,0.333333
3,S1-387694500,S3-789542813,Bombay Power Private Limited,Bombaypower.Com,"No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma ...","No: 6-32/2, 2Nd Floor, Mcc Towers, Opp To Sigm...",0.604651,0.558140,0.000000,0.960000,0.976744,0.863636,0.666667
4,S1-387694500,S3-9126403,Bombay Power Private Limited,ಬಾಂಬೆ ಪವರ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma ...","No: 6-32/2, Bangalore, KA",0.117647,0.127660,0.000000,0.291971,0.871795,0.181818,0.666667
5,S1-387694500,S3-904486811,Bombay Power Private Limited,ಬಾಂಬೆ ಪವರ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma ...","2Nd Floor, Mcc Towers, Opp To Sigma Soft Tech ...",0.117647,0.127660,0.000000,0.888889,0.976744,0.863636,0.666667
6,S1-387694500,S3-77197957,Bombay Power Private Limited,Bombay Private Límited Center,"No: 32/2, 2Nd Floor, Mcc Towers, Opp To Sigma ...",,0.736842,0.736842,0.333333,0.000000,0.000000,0.000000,0.000000
7,S1-881535873,S2-642463929,Azamgarh Marketing Private Limited,azamgarhmarketing.com,"C/O Ajai Kumar Rai, Village Lachirampur Hafizp...","PLOT 58 C/O AJAI KUMAR RAI, VILLAGE LACHIRAMPU...",0.690909,0.654545,0.000000,0.955556,1.000000,0.866667,0.000000
8,S1-436985273,S3-150259143,Corner Pilates,cornerpilates.com,"KY, 116 Redwood Drive, Frankfort","0116 Redwood Drive, Frankfort, Kentucky",0.838710,0.838710,0.000000,0.805970,0.895522,0.428571,0.000000
9,S1-254753234,S3-906680755,Interstate Products Inc,Center Interstate Inc,"Houston, TX, 11923 Gatlinburg Drive","Gatlinburg Drive, Houston, Texas",0.636364,0.800000,0.500000,0.507937,0.888889,0.500000,0.000000


In [110]:
thresholds = [0.8, 0.85, 0.9, 0.95]

for t in thresholds:
    mask = blocking_miss_features["address_token_set"] >= t
    print(
        f"address_token_set >= {t}: "
        f"{mask.sum()} / {len(mask)} "
        f"({mask.mean():.1%})"
    )

address_token_set >= 0.8: 69 / 89 (77.5%)
address_token_set >= 0.85: 62 / 89 (69.7%)
address_token_set >= 0.9: 57 / 89 (64.0%)
address_token_set >= 0.95: 38 / 89 (42.7%)


In [111]:
for t in [0.8, 0.85, 0.9, 0.95]:
    mask = blocking_miss_features["address_ratio"] >= t
    print(
        f"address_ratio >= {t}: "
        f"{mask.sum()} / {len(mask)} "
        f"({mask.mean():.1%})"
    )

address_ratio >= 0.8: 25 / 89 (28.1%)
address_ratio >= 0.85: 21 / 89 (23.6%)
address_ratio >= 0.9: 14 / 89 (15.7%)
address_ratio >= 0.95: 8 / 89 (9.0%)


In [112]:
mask = (
    (blocking_miss_features["address_token_set"] >= 0.8) &
    (blocking_miss_features["name_jaccard"] == 0)
)

print(
    "Strong address + zero name Jaccard:",
    mask.sum(),
    "/",
    len(mask)
)

Strong address + zero name Jaccard: 54 / 89


In [113]:
mask = (
    (blocking_miss_features["address_token_set"] >= 0.8) &
    (blocking_miss_features["name_ratio"] < 0.3)
)

print(
    "Strong address + name ratio < 0.3:",
    mask.sum(),
    "/",
    len(mask)
)

Strong address + name ratio < 0.3: 37 / 89


In [114]:
mask = (
    (blocking_miss_features["name_ratio"] < 0.3) &
    (blocking_miss_features["address_token_set"] >= 0.8)
)

print("Likely cross-script/address-dominant:", mask.sum())

Likely cross-script/address-dominant: 37


In [115]:
for threshold in [0.8, 0.9, 0.95, 1.0]:
    mask = (
        blocking_miss_features["address_token_set"] >= threshold
    )
    
    print(
        f"token_set >= {threshold}: "
        f"{mask.sum()} missed links"
    )

token_set >= 0.8: 69 missed links
token_set >= 0.9: 57 missed links
token_set >= 0.95: 38 missed links
token_set >= 1.0: 22 missed links


In [116]:
from src.preprocessing import normalize_address

miss_analysis["s1_addr_norm"] = (
    miss_analysis["s1_address"]
    .fillna("")
    .map(normalize_address)
)

miss_analysis["matched_addr_norm"] = (
    miss_analysis["matched_address"]
    .fillna("")
    .map(normalize_address)
)

miss_analysis["address_norm_exact"] = (
    miss_analysis["s1_addr_norm"]
    == miss_analysis["matched_addr_norm"]
)

print(
    "Exact normalized address:",
    miss_analysis["address_norm_exact"].sum(),
    "/",
    len(miss_analysis)
)

Exact normalized address: 3 / 89


In [117]:
print(
    miss_analysis[
        miss_analysis["address_norm_exact"]
    ][
        [
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address"
        ]
    ].head(20).to_string(index=False)
)

                                 s1_name                           matched_name                                                                                                             s1_address                                                                                                        matched_address
                         Precision Eaton                     Precision Services                                                                                          89 Central Avenue, Milton, MA                                                                                        ##89 Central Avenue, MILTON, MA
             Sai Finance Private Limited        ಸಾಯಿ ಫೈನಾನ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್ No 6, 2Nd Floor, Poorna Shashi Niwas, 4Th Cross, Gurudarshan Layout, Vidyarany, Apura, Bengaluru, Bangalore, Karnataka NO 6, 2ND FLOOR, POORNA SHASHI NIWAS, 4TH CROSS, GURUDARSHAN LAYOUT, VIDYARANY, APURA, BENGALURU, BANGALORE, Karnataka
One Unique Constructions Private Limited वन यू

In [118]:
for threshold in [0.5, 0.75, 1.0]:
    mask = blocking_miss_features["digit_overlap"] >= threshold
    print(
        f"digit_overlap >= {threshold}: "
        f"{mask.sum()} / {len(mask)} "
        f"({mask.mean():.1%})"
    )

digit_overlap >= 0.5: 41 / 89 (46.1%)
digit_overlap >= 0.75: 30 / 89 (33.7%)
digit_overlap >= 1.0: 26 / 89 (29.2%)


In [119]:
from sklearn.model_selection import train_test_split

s1_ids = candidate_features["s1_id"].unique()

train_s1, val_s1 = train_test_split(
    s1_ids,
    test_size=0.2,
    random_state=42
)

train_s1 = set(train_s1)
val_s1 = set(val_s1)

train_df = candidate_features[
    candidate_features["s1_id"].isin(train_s1)
].copy()

val_df = candidate_features[
    candidate_features["s1_id"].isin(val_s1)
].copy()

print("Train pairs:", len(train_df))
print("Validation pairs:", len(val_df))
print("Train S1:", len(train_s1))
print("Validation S1:", len(val_s1))

Train pairs: 796967
Validation pairs: 148967
Train S1: 400
Validation S1: 100


In [120]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

feature_cols = [
    "name_exact",
    "name_ratio",
    "name_token_set",
    "name_token_sort",
    "name_jaccard",
    "name_char_jaccard",
    "name_length_difference",
    "name_length_ratio",
    "address_exact",
    "address_ratio",
    "address_token_set",
    "address_token_sort",
    "address_jaccard",
    "address_char_jaccard",
    "address_length_difference",
    "address_length_ratio",
    "digit_overlap",
    "country_same",
    "both_exact",
]

X_train = train_df[feature_cols]
y_train = train_df["is_match"].astype(int)

X_val = val_df[feature_cols]
y_val = val_df["is_match"].astype(int)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]

print("Model trained.")

Model trained.


In [121]:
from sklearn.metrics import precision_score, recall_score, fbeta_score

results = []

for threshold in np.arange(0.05, 1.00, 0.05):

    preds = (val_probs >= threshold).astype(int)

    precision = precision_score(
        y_val,
        preds,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        preds,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val,
        preds,
        beta=0.5,
        zero_division=0
    )

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f0.5": f05
    })

threshold_results = pd.DataFrame(results)

threshold_results.sort_values(
    "f0.5",
    ascending=False
).head(10)

,threshold,precision,recall,f0.5
18,0.95,0.539855,0.949045,0.590801
17,0.90,0.453453,0.961783,0.507052
16,0.85,0.397638,0.964968,0.450625
15,0.80,0.359155,0.974522,0.411069
14,0.75,0.325902,0.977707,0.376041
13,0.70,0.299220,0.977707,0.347442
12,0.65,0.280365,0.977707,0.327013
11,0.60,0.262169,0.977707,0.307123
10,0.55,0.245797,0.977707,0.289077
9,0.50,0.229167,0.980892,0.270650


In [123]:
val_df = val_df.copy()
val_df["model_prob"] = val_probs

print(val_df[["s1_id", "matched_id", "model_prob"]].head())

          s1_id    matched_id    model_prob
0  S1-708796012  S2-100115999  2.822188e-08
1  S1-708796012  S2-101266972  7.614627e-06
2  S1-708796012  S2-101272680  8.483530e-05
3  S1-708796012  S2-102001891  7.485107e-03
4  S1-708796012  S2-103108266  1.050294e-04


In [124]:
best_pair_threshold = 0.95

predictions = {}

for s1_id, group in val_df.groupby("s1_id"):

    selected = group.loc[
        group["model_prob"] >= best_pair_threshold,
        "matched_id"
    ].tolist()

    predictions[s1_id] = selected

print("Validation S1 predictions:", len(predictions))

Validation S1 predictions: 100


In [129]:
print("model_prob" in val_df.columns)
print(val_df["model_prob"].describe())

True
count    1.489670e+05
mean     1.175148e-02
std      8.566673e-02
min      2.986820e-16
25%      1.613117e-06
50%      1.713529e-05
75%      2.011582e-04
max      1.000000e+00
Name: model_prob, dtype: float64


In [131]:
val_s1_set = set(val_s1)

val_ground_truth = {}

for chunk in pd.read_csv(
    DATA_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False,
    chunksize=200_000,
):
    chunk = chunk[
        chunk["source1_entity_id"].isin(val_s1_set)
    ]

    for _, row in chunk.iterrows():
        raw_matches = row["matched_entity_ids"]

        if not raw_matches:
            matches = []
        else:
            matches = [
                x.strip()
                for x in raw_matches.split(",")
                if x.strip()
            ]

        val_ground_truth[row["source1_entity_id"]] = matches

print("Validation GT S1:", len(val_ground_truth))
print(
    "Non-empty GT:",
    sum(bool(v) for v in val_ground_truth.values())
)

Validation GT S1: 100
Non-empty GT: 95


In [132]:
print(
    list(val_ground_truth.items())[:5]
)

[('S1-105568631', ['S2-743200797', 'S3-686929711']), ('S1-155025322', ['S2-720124786', 'S2-550739373', 'S3-725722380', 'S3-433685986']), ('S1-491641613', ['S2-844468416', 'S2-538252662', 'S3-231303629']), ('S1-348739044', ['S2-149870688', 'S3-665236884']), ('S1-105632026', ['S2-137595140', 'S2-477800212', 'S3-235164648', 'S3-365729049'])]


In [133]:
match_counts = pd.Series(
    {
        s1_id: len(matches)
        for s1_id, matches in val_ground_truth.items()
    }
)

print(match_counts.describe())
print("\nDistribution:")
print(match_counts.value_counts().sort_index())

count    100.000000
mean       3.370000
std        1.807141
min        0.000000
25%        2.000000
50%        3.000000
75%        4.000000
max       10.000000
dtype: float64

Distribution:
0      5
1      7
2     19
3     27
4     20
5     10
6      7
7      3
8      1
10     1
Name: count, dtype: int64


In [135]:
from src.evaluation import entity_f05

In [136]:
threshold_results_entity = []

for threshold in np.arange(0.50, 1.00, 0.025):

    predictions = {}

    for s1_id, group in val_df.groupby("s1_id"):

        selected = group.loc[
            group["model_prob"] >= threshold,
            "matched_id"
        ].tolist()

        predictions[s1_id] = selected

    scores = []

    for s1_id in val_s1:

        actual = val_ground_truth.get(s1_id, [])
        predicted = predictions.get(s1_id, [])

        scores.append(
            entity_f05(predicted, actual)
        )

    threshold_results_entity.append({
        "threshold": threshold,
        "macro_f0.5": np.mean(scores),
    })

entity_threshold_results = pd.DataFrame(
    threshold_results_entity
)

print(
    entity_threshold_results
    .sort_values("macro_f0.5", ascending=False)
    .head(10)
)

    threshold  macro_f0.5
19      0.975    0.729590
18      0.950    0.700142
17      0.925    0.665571
16      0.900    0.657573
15      0.875    0.631376
14      0.850    0.616510
13      0.825    0.593068
12      0.800    0.580494
11      0.775    0.559655
10      0.750    0.552980


In [137]:
best_threshold = 0.975
best_macro_f05 = 0.729590

print("Baseline threshold:", best_threshold)
print("Baseline validation Macro F0.5:", best_macro_f05)

Baseline threshold: 0.975
Baseline validation Macro F0.5: 0.72959


In [138]:
entity_threshold_results.to_csv(
    ROOT / "experiments" / "baseline_threshold_results.csv",
    index=False
)

In [139]:
best_threshold = 0.975

best_predictions = {}

for s1_id, group in val_df.groupby("s1_id"):
    selected = group.loc[
        group["model_prob"] >= best_threshold,
        "matched_id"
    ].tolist()

    best_predictions[s1_id] = selected

In [140]:
pred_counts = pd.Series({
    s1_id: len(matches)
    for s1_id, matches in best_predictions.items()
})

gt_counts = pd.Series({
    s1_id: len(val_ground_truth.get(s1_id, []))
    for s1_id in val_s1
})

count_comparison = pd.DataFrame({
    "predicted": pred_counts,
    "actual": gt_counts
}).fillna(0)

print(count_comparison.describe())

print("\nPredicted match-count distribution:")
print(pred_counts.value_counts().sort_index())

print("\nActual match-count distribution:")
print(gt_counts.value_counts().sort_index())

        predicted      actual
count  100.000000  100.000000
mean     4.740000    3.370000
std      3.572114    1.807141
min      0.000000    0.000000
25%      3.000000    2.000000
50%      4.000000    3.000000
75%      6.000000    4.000000
max     26.000000   10.000000

Predicted match-count distribution:
0      2
1      8
2     12
3     17
4     18
5     17
6      7
7      6
8      4
9      3
11     2
12     1
15     1
16     1
26     1
Name: count, dtype: int64

Actual match-count distribution:
0      5
1      7
2     19
3     27
4     20
5     10
6      7
7      3
8      1
10     1
Name: count, dtype: int64


In [141]:
error_rows = []

for s1_id in val_s1:

    actual = set(val_ground_truth.get(s1_id, []))
    predicted = set(best_predictions.get(s1_id, []))

    false_positive_ids = predicted - actual
    false_negative_ids = actual - predicted

    error_rows.append({
        "s1_id": s1_id,
        "actual_count": len(actual),
        "predicted_count": len(predicted),
        "false_positive_count": len(false_positive_ids),
        "false_negative_count": len(false_negative_ids),
        "false_positives": list(false_positive_ids),
        "false_negatives": list(false_negative_ids),
    })

error_df = pd.DataFrame(error_rows)

print(
    "S1s with false positives:",
    (error_df["false_positive_count"] > 0).sum()
)

print(
    "S1s with false negatives:",
    (error_df["false_negative_count"] > 0).sum()
)

print(
    "Total false positives:",
    error_df["false_positive_count"].sum()
)

print(
    "Total false negatives:",
    error_df["false_negative_count"].sum()
)

S1s with false positives: 54
S1s with false negatives: 28
Total false positives: 180
Total false negatives: 43


In [142]:
fp_pairs = []

for s1_id, group in val_df.groupby("s1_id"):

    actual = set(val_ground_truth.get(s1_id, []))

    selected = group[
        group["model_prob"] >= best_threshold
    ]

    for _, row in selected.iterrows():

        if row["matched_id"] not in actual:
            fp_pairs.append(row)

false_positive_pairs = pd.DataFrame(fp_pairs)

print("False-positive pairs:", len(false_positive_pairs))

False-positive pairs: 180


In [143]:
fn_pairs = []

for s1_id in val_s1:

    actual = set(val_ground_truth.get(s1_id, []))

    group = val_df[
        val_df["s1_id"] == s1_id
    ]

    predicted = set(
        group.loc[
            group["model_prob"] >= best_threshold,
            "matched_id"
        ]
    )

    for matched_id in actual - predicted:

        row = group[
            group["matched_id"] == matched_id
        ]

        if not row.empty:
            fn_pairs.append(row.iloc[0])

false_negative_pairs = pd.DataFrame(fn_pairs)

print("False-negative pairs:", len(false_negative_pairs))

False-negative pairs: 20


In [144]:
error_feature_summary = pd.DataFrame({
    "false_positive_mean": false_positive_pairs[feature_cols].mean(),
    "false_negative_mean": false_negative_pairs[feature_cols].mean(),
})

error_feature_summary["difference"] = (
    error_feature_summary["false_negative_mean"]
    - error_feature_summary["false_positive_mean"]
)

error_feature_summary.sort_values(
    "difference",
    ascending=False
)

,false_positive_mean,false_negative_mean,difference
address_length_difference,23.772222,39.250000,15.477778
name_length_difference,4.550000,4.950000,0.400000
address_exact,0.005556,0.100000,0.094444
name_exact,0.038889,0.050000,0.011111
both_exact,0.000000,0.000000,0.000000
country_same,1.000000,1.000000,0.000000
name_length_ratio,0.835080,0.822731,-0.012349
name_char_jaccard,0.366679,0.346769,-0.019911
name_token_sort,0.579684,0.544976,-0.034708
name_jaccard,0.357928,0.322143,-0.035785


In [145]:
false_positive_pairs[
    [
        "s1_id",
        "matched_id",
        "model_prob",
        "name_ratio",
        "name_token_set",
        "name_jaccard",
        "name_char_jaccard",
        "address_ratio",
        "address_token_set",
        "address_jaccard",
        "address_char_jaccard",
        "digit_overlap",
        "country_same",
    ]
].sort_values(
    "model_prob",
    ascending=False
).head(20)

,s1_id,matched_id,model_prob,name_ratio,name_token_set,name_jaccard,name_char_jaccard,address_ratio,address_token_set,address_jaccard,address_char_jaccard,digit_overlap,country_same
336899,S1-336028803,S2-555996801,1.000000,0.772727,0.954545,0.500000,0.600000,0.894118,0.980132,0.882353,0.867470,0.666667,1
84799,S1-417600197,S3-141091672,1.000000,0.864198,1.000000,0.833333,0.704545,0.949153,0.957576,0.777778,0.855556,0.500000,1
933995,S1-362026428,S2-910900313,1.000000,0.645161,0.697674,0.333333,0.348837,0.873016,1.000000,0.769231,0.738462,1.000000,1
892170,S1-801681033,S2-586722066,0.999999,0.722222,0.857143,0.500000,0.523810,0.566038,0.981132,0.666667,0.620690,1.000000,1
203980,S1-148555813,S3-11985184,0.999999,0.285714,0.244898,0.000000,0.000000,0.914286,0.950820,0.666667,0.833333,1.000000,1
181737,S1-750136146,S3-947215660,0.999998,0.842105,0.842105,0.500000,0.565217,0.927152,0.942149,0.733333,0.824324,0.333333,1
840143,S1-562380004,S2-958892154,0.999987,0.836364,1.000000,0.800000,0.689655,0.621622,0.972973,0.714286,0.707317,0.000000,1
729019,S1-213245163,S3-580713557,0.999984,0.885714,1.000000,0.800000,0.783784,0.803150,0.803150,0.272727,0.556962,0.000000,1
713599,S1-265022480,S3-386441982,0.999981,0.690909,0.727273,0.333333,0.378378,0.814815,0.888889,0.454545,0.604167,0.333333,1
774732,S1-363688745,S3-108574819,0.999974,0.342857,0.342857,0.000000,0.000000,0.571429,0.961039,0.714286,0.738095,1.000000,1


In [146]:
false_negative_pairs[
    [
        "s1_id",
        "matched_id",
        "model_prob",
        "name_ratio",
        "name_token_set",
        "name_jaccard",
        "name_char_jaccard",
        "address_ratio",
        "address_token_set",
        "address_jaccard",
        "address_char_jaccard",
        "digit_overlap",
        "country_same",
    ]
].sort_values(
    "model_prob",
    ascending=False
).head(20)

,s1_id,matched_id,model_prob,name_ratio,name_token_set,name_jaccard,name_char_jaccard,address_ratio,address_token_set,address_jaccard,address_char_jaccard,digit_overlap,country_same
829805,S1-768534635,S2-626226436,0.965202,0.898551,0.898551,0.500000,0.710526,0.000000,0.000000,0.000000,0.000000,0.0,1
662936,S1-155025322,S3-433685986,0.964302,0.826087,1.000000,0.666667,0.652174,0.000000,0.000000,0.000000,0.000000,0.0,1
180288,S1-750136146,S2-936251078,0.958086,0.933333,0.933333,0.600000,0.750000,0.000000,0.000000,0.000000,0.000000,0.0,1
376190,S1-989596134,S3-499848683,0.950489,0.742857,0.971429,0.500000,0.631579,0.492754,0.666667,0.083333,0.207547,1.0,1
32991,S1-851769617,S2-98185691,0.929305,1.000000,1.000000,1.000000,1.000000,0.621622,0.648649,0.166667,0.250000,0.0,1
165885,S1-648674437,S2-219572257,0.919753,0.089552,0.105263,0.000000,0.000000,0.483871,0.736842,0.384615,0.247191,1.0,1
84512,S1-417600197,S2-902738017,0.919066,0.870968,1.000000,0.800000,0.750000,0.000000,0.000000,0.000000,0.000000,0.0,1
70728,S1-807703451,S3-784007691,0.918919,0.976744,0.976744,0.500000,0.772727,0.000000,0.000000,0.000000,0.000000,0.0,1
159054,S1-629879582,S3-117318892,0.891476,0.473684,0.588235,0.142857,0.254545,0.611465,0.891892,0.300000,0.310680,0.0,1
685441,S1-538846322,S2-403925239,0.823950,0.109091,0.122449,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.0,1


In [147]:
# Which false negatives were actually present in the candidate set?
candidate_fn_count = len(false_negative_pairs)

# Total FN links from S1-level comparison
total_fn = error_df["false_negative_count"].sum()

print("Total false-negative links:", total_fn)
print("FN links present in candidate set:", candidate_fn_count)
print("FN links missing from candidate set:", total_fn - candidate_fn_count)

print(
    "Fraction of FN links caused by blocking:",
    round((total_fn - candidate_fn_count) / total_fn, 4)
)

Total false-negative links: 43
FN links present in candidate set: 20
FN links missing from candidate set: 23
Fraction of FN links caused by blocking: 0.5349


In [148]:
fp_summary = false_positive_pairs[feature_cols].describe().T

fp_summary["mean"] = false_positive_pairs[feature_cols].mean()
fp_summary = fp_summary.sort_values("mean", ascending=False)

fp_summary[["mean", "50%", "75%", "max"]]

,mean,50%,75%,max
address_length_difference,23.772222,23.000000,37.000000,94.0
name_length_difference,4.550000,4.000000,6.250000,24.0
country_same,1.000000,1.000000,1.000000,1.0
name_length_ratio,0.835080,0.856034,0.944820,1.0
name_token_set,0.654351,0.714286,0.914512,1.0
address_token_set,0.603918,0.825070,0.935655,1.0
name_ratio,0.594354,0.643462,0.818554,1.0
name_token_sort,0.579684,0.578201,0.800000,1.0
address_token_sort,0.515341,0.603571,0.869565,1.0
address_length_ratio,0.511993,0.625000,0.887847,1.0


In [149]:
print("False-positive probability distribution:")
print(
    false_positive_pairs["model_prob"]
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)

False-positive probability distribution:
count    180.000000
mean       0.992079
std        0.007934
min        0.975837
50%        0.995570
75%        0.999157
90%        0.999944
95%        0.999975
99%        1.000000
max        1.000000
Name: model_prob, dtype: float64


In [150]:
fp_display_cols = [
    "s1_id",
    "matched_id",
    "model_prob",
    "name_exact",
    "name_ratio",
    "name_token_set",
    "name_jaccard",
    "name_char_jaccard",
    "address_exact",
    "address_ratio",
    "address_token_set",
    "address_jaccard",
    "address_char_jaccard",
    "digit_overlap",
    "country_same",
]

false_positive_pairs.sort_values(
    "model_prob", ascending=False
)[fp_display_cols].head(25)

,s1_id,matched_id,model_prob,name_exact,name_ratio,name_token_set,name_jaccard,name_char_jaccard,address_exact,address_ratio,address_token_set,address_jaccard,address_char_jaccard,digit_overlap,country_same
336899,S1-336028803,S2-555996801,1.000000,0,0.772727,0.954545,0.500000,0.600000,0,0.894118,0.980132,0.882353,0.867470,0.666667,1
84799,S1-417600197,S3-141091672,1.000000,0,0.864198,1.000000,0.833333,0.704545,0,0.949153,0.957576,0.777778,0.855556,0.500000,1
933995,S1-362026428,S2-910900313,1.000000,0,0.645161,0.697674,0.333333,0.348837,0,0.873016,1.000000,0.769231,0.738462,1.000000,1
892170,S1-801681033,S2-586722066,0.999999,0,0.722222,0.857143,0.500000,0.523810,0,0.566038,0.981132,0.666667,0.620690,1.000000,1
203980,S1-148555813,S3-11985184,0.999999,0,0.285714,0.244898,0.000000,0.000000,0,0.914286,0.950820,0.666667,0.833333,1.000000,1
181737,S1-750136146,S3-947215660,0.999998,0,0.842105,0.842105,0.500000,0.565217,0,0.927152,0.942149,0.733333,0.824324,0.333333,1
840143,S1-562380004,S2-958892154,0.999987,0,0.836364,1.000000,0.800000,0.689655,0,0.621622,0.972973,0.714286,0.707317,0.000000,1
729019,S1-213245163,S3-580713557,0.999984,0,0.885714,1.000000,0.800000,0.783784,0,0.803150,0.803150,0.272727,0.556962,0.000000,1
713599,S1-265022480,S3-386441982,0.999981,0,0.690909,0.727273,0.333333,0.378378,0,0.814815,0.888889,0.454545,0.604167,0.333333,1
774732,S1-363688745,S3-108574819,0.999974,0,0.342857,0.342857,0.000000,0.000000,0,0.571429,0.961039,0.714286,0.738095,1.000000,1


In [151]:
fn_display_cols = [
    "s1_id",
    "matched_id",
    "model_prob",
    "name_exact",
    "name_ratio",
    "name_token_set",
    "name_jaccard",
    "name_char_jaccard",
    "address_exact",
    "address_ratio",
    "address_token_set",
    "address_jaccard",
    "address_char_jaccard",
    "digit_overlap",
    "country_same",
]

false_negative_pairs.sort_values(
    "model_prob", ascending=False
)[fn_display_cols]

,s1_id,matched_id,model_prob,name_exact,name_ratio,name_token_set,name_jaccard,name_char_jaccard,address_exact,address_ratio,address_token_set,address_jaccard,address_char_jaccard,digit_overlap,country_same
829805,S1-768534635,S2-626226436,0.965202,0,0.898551,0.898551,0.500000,0.710526,0,0.000000,0.000000,0.000000,0.000000,0.0,1
662936,S1-155025322,S3-433685986,0.964302,0,0.826087,1.000000,0.666667,0.652174,0,0.000000,0.000000,0.000000,0.000000,0.0,1
180288,S1-750136146,S2-936251078,0.958086,0,0.933333,0.933333,0.600000,0.750000,0,0.000000,0.000000,0.000000,0.000000,0.0,1
376190,S1-989596134,S3-499848683,0.950489,0,0.742857,0.971429,0.500000,0.631579,0,0.492754,0.666667,0.083333,0.207547,1.0,1
32991,S1-851769617,S2-98185691,0.929305,1,1.000000,1.000000,1.000000,1.000000,0,0.621622,0.648649,0.166667,0.250000,0.0,1
165885,S1-648674437,S2-219572257,0.919753,0,0.089552,0.105263,0.000000,0.000000,0,0.483871,0.736842,0.384615,0.247191,1.0,1
84512,S1-417600197,S2-902738017,0.919066,0,0.870968,1.000000,0.800000,0.750000,0,0.000000,0.000000,0.000000,0.000000,0.0,1
70728,S1-807703451,S3-784007691,0.918919,0,0.976744,0.976744,0.500000,0.772727,0,0.000000,0.000000,0.000000,0.000000,0.0,1
159054,S1-629879582,S3-117318892,0.891476,0,0.473684,0.588235,0.142857,0.254545,0,0.611465,0.891892,0.300000,0.310680,0.0,1
685441,S1-538846322,S2-403925239,0.823950,0,0.109091,0.122449,0.000000,0.000000,1,1.000000,1.000000,1.000000,1.000000,0.0,1


In [152]:
print(
    false_negative_pairs[feature_cols]
    .mean()
    .sort_values(ascending=False)
)

address_length_difference    39.250000
name_length_difference        4.950000
country_same                  1.000000
name_length_ratio             0.822731
name_token_set                0.597928
name_ratio                    0.545407
name_token_sort               0.544976
address_token_set             0.484125
address_length_ratio          0.432961
address_token_sort            0.402485
address_ratio                 0.388939
name_char_jaccard             0.346769
name_jaccard                  0.322143
address_jaccard               0.251457
digit_overlap                 0.250000
address_char_jaccard          0.235528
address_exact                 0.100000
name_exact                    0.050000
both_exact                    0.000000
dtype: float64


In [153]:
error_feature_summary = pd.DataFrame({
    "false_positive_mean": false_positive_pairs[feature_cols].mean(),
    "false_negative_mean": false_negative_pairs[feature_cols].mean(),
})

error_feature_summary["difference"] = (
    error_feature_summary["false_negative_mean"]
    - error_feature_summary["false_positive_mean"]
)

error_feature_summary.sort_values(
    "difference",
    ascending=False
)

,false_positive_mean,false_negative_mean,difference
address_length_difference,23.772222,39.250000,15.477778
name_length_difference,4.550000,4.950000,0.400000
address_exact,0.005556,0.100000,0.094444
name_exact,0.038889,0.050000,0.011111
both_exact,0.000000,0.000000,0.000000
country_same,1.000000,1.000000,0.000000
name_length_ratio,0.835080,0.822731,-0.012349
name_char_jaccard,0.366679,0.346769,-0.019911
name_token_sort,0.579684,0.544976,-0.034708
name_jaccard,0.357928,0.322143,-0.035785
